# 1. MISALIGNMENT ROBUSTNESS — does the temporal mechanism help once misalignment exists? — AeroSync-Mamba v25

**Question.** v22 answered a question about clean data: HTT's time embedding and LACA's lag bias did not improve
accuracy, in any combination, over 5 seeds. The TII data, however, has little genuine cross-modal offset to learn
from. v25 asks something different. **Does Condition C (HTT time + LACA lag) gain an advantage over Condition D (no
temporal mechanism) once cross-modal misalignment is present at test time, and does that advantage grow with the size
of the misalignment?**

**Data and models.**
- **Data:** v22's data unchanged (t = 0-anchored audio, 99 missions, v18 preprocessing). The corrected alignment from
  v23/v24 is deliberately not used.
- **Models:** Condition C (HTT time ON, LACA lag β = 1.0) and Condition D (both OFF), 5 seeds each (42, 123, 2024, 7, 99).
- **Part 1: checkpoints.** v22 saved `v22_checkpoints/{C,D}_seed{s}_best.pt` to `/kaggle/working`. A checkpoint is
  reused if it is attached as a Kaggle input **and** reproduces v22's logged clean test result (±0.0005 on all 4
  metrics, identical confusion matrix). Otherwise that run is **retrained** with v22's unchanged `train_one_run`, and
  checked against the same numbers. v25 saves its own checkpoints to `v25_checkpoints/`, so a later run can reuse them.

**Part 2: test-time misalignment, evaluation only.** Two types of misalignment are each applied through **two arms**.

| | Type 1: fixed delay δ ∈ {0.05, 0.10, 0.20, 0.30} | Type 2: jitter σ ∈ {0.02, 0.05, 0.10, 0.20} |
|---|---|---|
| **Arm 1: τ-only** (your spec; the H2 mechanism) | τ_s ← clamp(τ_s + δ, 0, 1) | τ_s ← clamp(τ_s + σ·z_w, 0, 1) |
| **Arm 2: content shift** (real misalignment) | IMU window re-sliced from [t + δW, t + δW + W]; τ_s = the samples' TRUE times relative to the audio window (unclamped) | the same with shift σ·z_w·W |

Units: shifts are fractions of the window, W = 2.56 s. **z_w** is one standard-normal draw per test window, from
`np.random.default_rng(JITTER_SEED)`. That generator is separate from every training seed. The same z is used for
every σ, both arms and all 10 models, so within a jitter level the model is the only thing that changes. Each noise
vector's sha1 is logged and checked to be identical across models.

**Why two arms.** τ-only moves **no signal**. The audio and IMU content stays aligned, and the model is simply given
wrong timestamps. D never reads timestamps, so under τ-only it is exactly flat by construction, while C can only be
disturbed. The C − D gap therefore cannot grow in this arm. It is kept as specified, as a check of the injection code
and a replication of H2. Arm 2 creates **real** misalignment: the IMU content really comes from a different time than
the audio, and C is told the true timing. D sees the same misaligned content with no timing information. This is the
arm that can answer the question.

**Two details of Arm 2.**
- **Test windows.** It scores only the test windows whose shifted IMU window stays inside the flight at **every**
  magnitude, so the window set is the same at every magnitude. It has its own 0.0 baseline on that subset. At shift 0
  the re-slicer must reproduce v18's test windows bit for bit, which the notebook checks.
- **Timestamps.** Honest τ_s is not clamped, since clamping would erase the true timing. With shifts, τ_s lies in
  [δ, 1 + δ], which is outside the [0, 1] range seen in training.

**Part 3: the key comparison.** At every (arm, type, magnitude), the table reports:
- the paired difference C − D in macro-F1 over the 5 seeds: mean, SD, wins out of 5 and a paired t-test;
- the trend: gap at the largest magnitude minus gap at 0, per seed, plus a per-seed linear slope.

A growing gap counts as **real** only if both hold:
- the mean change in the gap exceeds its SD;
- at the largest magnitude, the mean C − D is positive and exceeds its SD.

**Secondary diagnostics.**
- Curves for C and D in every arm and type.
- Under τ-only, D must be **bit-identical** to its own zero-misalignment result at every magnitude of both types. If it
  is not, the notebook flags it loudly as an injection bug.

**Outputs:**
- `misalignment_robustness_results.csv`: one row per evaluation, written after each one.
- `misalignment_robustness_training.csv`: one row per obtained model, reused or retrained.


## 2. Configuration constants & environment
Every fixed value lives in the first cell below. The second installs `mamba_ssm` and reports which fusion backbone is active.

In [1]:
import copy
import csv
import gc
import hashlib
import importlib.util
import json
import math
import os
import random
import subprocess
import sys
import time
import urllib.request
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# ==============================================================================
# FIXED CONFIGURATION — every finalized choice (see the summary table above)
# ==============================================================================
# --- 2x2 factorial: HTT time embedding x LACA lag bias, 5 seeds, grouped by seed ---
SEEDS = [42, 123, 2024, 7, 99]           # same 5 seeds as every prior multi-seed experiment
CONDITIONS = {                           # name: (HTT_TIME_EMBED_ENABLED, LACA_LAG_BIAS_ENABLED)
    "A": (True, False),                  # control = v18 at beta = 0
    "B": (False, True),
    "C": (True, True),
    "D": (False, False),
}
# --- v25: misalignment robustness of Condition C vs Condition D (evaluation-time injection only) ---
EVAL_CONDITIONS = ["C", "D"]             # C = HTT-time ON + LACA-lag ON (beta 1.0); D = both OFF
RUN_ORDER = [(seed, cond) for seed in SEEDS for cond in EVAL_CONDITIONS]   # C, D per seed = 10 models
H2_OFFSETS = [0.0, 0.05, 0.10, 0.20, 0.30]   # v22's H2 offsets (kept: v22's h2_sweep is still defined in the training cell)
DELAYS = [0.0, 0.05, 0.10, 0.20, 0.30]   # TYPE 1: fixed systematic delay, in window fractions (= H2)
JITTER_SIGMAS = [0.0, 0.02, 0.05, 0.10, 0.20]   # TYPE 2: per-window Gaussian jitter SD, in window fractions
JITTER_SEED = 2025                       # seed of the jitter noise ONLY (own numpy Generator; never touches training RNG)
ARMS = ["tau_only", "content_shift"]     # tau_only = the H2 mechanism (spec); content_shift = real misalignment
SPLIT_SEED = 42                          # mission-level train/val/test split (independent of the training seed)

# --- Dataset / split ---
DATASET_REPO_URL = "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"
WORK_DIR = Path("/kaggle/working")
DATASET_BASE = WORK_DIR / "tii_uav_dataset"
DATASET_ROOT = DATASET_BASE / "Dataset"
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
N_CLASSES = 5                            # propeller-fault severity 0..4

# --- Windowing ---
WINDOW_SEC = 2.56
STEP_SEC = 1.28
MIN_SENSOR_SAMPLES_PER_WINDOW = 10

# --- Audio: strongest-64 bins (train-only), sqrt(|rFFT|), no window, no normalization ---
N_AUDIO_BINS = 64                        # frequency axis
TA_AUDIO = 64                            # audio time positions per window

# --- Sensor: raw 6 channels (accel x/y/z, gyro x/y/z), no derived features ---
N_SENSOR_CHANNELS = 6
TS_SENSOR = 128                          # sensor time positions per window
SENSOR_STD_EPS = 1e-6

# --- Model ---
D_MODEL = 256
TIME_EMBED_FREQUENCIES = 64
LACA_NUM_HEADS = 4
LACA_BETA = 1.0                          # beta when LACA_LAG_BIAS_ENABLED (fixed from epoch 1: no warm-up, not learnable)
LAG_BIAS_HIDDEN = 32                     # mlp g(Δτ): Linear(1,32) -> Tanh -> Linear(32,1) = 97 params
LAG_BIAS_PARAMS = 97                     # the LayerNorm added in v18 has no parameters -> still 97
LAG_BIAS_LN_EPS = 1e-5                   # LayerNorm over each window's (T_m, T_n) lag-bias grid, no affine
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496          # 2 x 437,248 (real mamba_ssm)
DROPOUT = 0.15

# --- Loss: L_cls (class-weighted CE) + 0.1 * L_align (InfoNCE) + 0.1 * L_cons (KL) ---
LAMBDA_ALIGN = 0.1
LAMBDA_CONS = 0.1
INFONCE_TEMPERATURE = 0.07

# --- Optimization ---
EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2

# --- Outputs ---
CHECKPOINT_DIR = WORK_DIR / "v25_checkpoints"   # retrained runs are saved here (attach them to skip retraining next time)
PROBS_DIR = WORK_DIR / "v25_test_probs"
RESULTS_CSV = WORK_DIR / "misalignment_robustness_results.csv"
TRAINING_CSV = WORK_DIR / "misalignment_robustness_training.csv"
# Where v22 (or an earlier v25) checkpoints may be: attached Kaggle inputs, or this session's working dir.
CHECKPOINT_GLOBS = ["/kaggle/input/**/v22_checkpoints/{name}", "/kaggle/input/**/v25_checkpoints/{name}",
                    str(WORK_DIR / "v22_checkpoints" / "{name}"), str(WORK_DIR / "v25_checkpoints" / "{name}")]
for _d in (CHECKPOINT_DIR, PROBS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- Condition A candidates for reuse: v18's beta=0.0 runs (HTT time embedding always ON in v18) ---
# 4 metrics + init / RNG fingerprints: Artifacts/Version 18 CSV. Confusion matrices: Artifacts/Version 20 CSV (the same
# runs, regenerated in v20 with all 4 metrics identical to v18).
V18_BETA0_RUNS = {
    42: {"accuracy": 0.9088495575221239, "macro_f1": 0.9057330017972403, "balanced_accuracy": 0.9030033658152373, "macro_auroc": 0.9891867537737307,
          "confusion_matrix": [[283, 0, 1, 0, 2], [3, 197, 2, 1, 2], [0, 1, 172, 35, 2], [0, 0, 8, 204, 16], [0, 0, 2, 28, 171]],
          "init_sha1": "5850841e699f", "rng_sha1": "5641b401720d"},
    123: {"accuracy": 0.8876106194690265, "macro_f1": 0.8826825575079713, "balanced_accuracy": 0.8790467961967972, "macro_auroc": 0.9861866752615566,
          "confusion_matrix": [[283, 1, 1, 0, 1], [2, 196, 6, 1, 0], [0, 2, 177, 30, 1], [0, 0, 8, 203, 17], [0, 1, 2, 54, 144]],
          "init_sha1": "74be0883c62b", "rng_sha1": "0a4b3be44b18"},
    2024: {"accuracy": 0.8805309734513275, "macro_f1": 0.8763643000413779, "balanced_accuracy": 0.8715485268344848, "macro_auroc": 0.9866958753709998,
          "confusion_matrix": [[282, 1, 0, 0, 3], [2, 189, 12, 1, 1], [0, 0, 168, 39, 3], [0, 0, 6, 206, 16], [0, 0, 0, 51, 150]],
          "init_sha1": "92852955eef1", "rng_sha1": "619c0e0a6b4d"},
    7: {"accuracy": 0.9061946902654867, "macro_f1": 0.9019350016739314, "balanced_accuracy": 0.9008047135870789, "macro_auroc": 0.9897445660698647,
          "confusion_matrix": [[284, 1, 0, 0, 1], [2, 196, 7, 0, 0], [0, 3, 183, 22, 2], [0, 0, 16, 191, 21], [0, 0, 3, 28, 170]],
          "init_sha1": "7f5d0cd2e7cb", "rng_sha1": "9fdcf777d285"},
    99: {"accuracy": 0.8690265486725663, "macro_f1": 0.8638026478536822, "balanced_accuracy": 0.8598248510633313, "macro_auroc": 0.9829841151452052,
          "confusion_matrix": [[282, 0, 1, 0, 3], [5, 190, 9, 0, 1], [0, 2, 164, 39, 5], [0, 0, 12, 197, 19], [0, 1, 3, 48, 149]],
          "init_sha1": "e40c78169f57", "rng_sha1": "81dcd4638a88"},
}
SAME_SEED_TOLERANCE = 0.0005             # same-seed reproducibility (code correctness), not a finding threshold
SEED_TO_SEED_STD_F1 = 0.0160             # v15/v16 measured seed-to-seed std of test macro-F1
PAIRED_ALPHA = 0.05
REFERENCE_SEED42_RNG_PREFIX = "5641b401720d"   # torch CPU RNG right after seed-42 model construction (v10-v15)
REFERENCE_DATA = {
    "bin_indices": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255,
                    258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508,
                    509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760,
                    761, 762, 763, 764, 765, 766],
    "sensor_fingerprint": {
        "train": (4934, -6167105.225, 808427.456, (906, 1020, 1004, 1008, 996)),
        "val":   (954, -1193206.652, 156301.664, (126, 229, 199, 190, 210)),
        "test":  (1130, -1413563.944, 185127.438, (286, 205, 210, 228, 201)),
    },
}

# --- v25 references: v22's logged clean test results for Conditions C and D (Artifacts/Version 22 CSV) ---
V22_CD_RUNS = {
    'C42': {'accuracy': 0.8938053097345132, 'macro_f1': 0.8900430576358016, 'balanced_accuracy': 0.8873790660511102, 'macro_auroc': 0.986928283391767, 'class3_recall': 0.8640350877192983, 'class4_recall': 0.8258706467661692, 'confusion_matrix': [[282, 0, 1, 0, 3], [2, 197, 4, 0, 2], [0, 2, 168, 39, 1], [0, 0, 9, 197, 22], [0, 0, 3, 32, 166]]},
    'D42': {'accuracy': 0.9097345132743363, 'macro_f1': 0.90640272771588, 'balanced_accuracy': 0.9040236181473512, 'macro_auroc': 0.9890678208140686, 'class3_recall': 0.8859649122807017, 'class4_recall': 0.8706467661691543, 'confusion_matrix': [[284, 0, 0, 0, 2], [3, 197, 4, 0, 1], [0, 1, 170, 38, 1], [0, 0, 10, 202, 16], [0, 0, 3, 23, 175]]},
    'C123': {'accuracy': 0.8884955752212389, 'macro_f1': 0.8838406391126263, 'balanced_accuracy': 0.880942616928094, 'macro_auroc': 0.9853528858095087, 'class3_recall': 0.8640350877192983, 'class4_recall': 0.7711442786069652, 'confusion_matrix': [[283, 0, 1, 0, 2], [2, 197, 5, 0, 1], [0, 4, 172, 32, 2], [0, 0, 7, 197, 24], [0, 0, 3, 43, 155]]},
    'D123': {'accuracy': 0.8991150442477877, 'macro_f1': 0.894796448034539, 'balanced_accuracy': 0.8915204180283789, 'macro_auroc': 0.9882101818367766, 'class3_recall': 0.8991228070175439, 'class4_recall': 0.7661691542288557, 'confusion_matrix': [[284, 1, 1, 0, 0], [1, 199, 4, 0, 1], [0, 4, 174, 32, 0], [0, 0, 8, 205, 15], [1, 0, 2, 44, 154]]},
    'C2024': {'accuracy': 0.879646017699115, 'macro_f1': 0.8742357339300486, 'balanced_accuracy': 0.8720160236240938, 'macro_auroc': 0.9862867775865294, 'class3_recall': 0.8157894736842105, 'class4_recall': 0.7711442786069652, 'confusion_matrix': [[284, 0, 0, 0, 2], [2, 198, 4, 0, 1], [0, 5, 171, 33, 1], [0, 0, 13, 186, 29], [0, 1, 4, 41, 155]]},
    'D2024': {'accuracy': 0.8867256637168142, 'macro_f1': 0.8824386737488268, 'balanced_accuracy': 0.8794072629163541, 'macro_auroc': 0.9864377660864745, 'class3_recall': 0.8464912280701754, 'class4_recall': 0.8059701492537313, 'confusion_matrix': [[284, 1, 0, 0, 1], [0, 198, 6, 0, 1], [0, 2, 165, 42, 1], [0, 0, 11, 193, 24], [0, 0, 2, 37, 162]]},
    'C7': {'accuracy': 0.8964601769911504, 'macro_f1': 0.8920671005070464, 'balanced_accuracy': 0.890241547389824, 'macro_auroc': 0.9884796293166703, 'class3_recall': 0.8421052631578947, 'class4_recall': 0.8308457711442786, 'confusion_matrix': [[284, 1, 1, 0, 0], [1, 201, 3, 0, 0], [0, 5, 169, 35, 1], [0, 0, 20, 192, 16], [0, 1, 2, 31, 167]]},
    'D7': {'accuracy': 0.8876106194690265, 'macro_f1': 0.8835475452484998, 'balanced_accuracy': 0.8803460855559797, 'macro_auroc': 0.9863885321796735, 'class3_recall': 0.8771929824561403, 'class4_recall': 0.8308457711442786, 'confusion_matrix': [[283, 1, 1, 0, 1], [1, 200, 4, 0, 0], [0, 0, 153, 53, 4], [0, 0, 7, 200, 21], [1, 0, 1, 32, 167]]},
    'C99': {'accuracy': 0.863716814159292, 'macro_f1': 0.8577105576939189, 'balanced_accuracy': 0.8553663166676406, 'macro_auroc': 0.9842761673528237, 'class3_recall': 0.7982456140350878, 'class4_recall': 0.7562189054726368, 'confusion_matrix': [[282, 1, 1, 0, 2], [4, 190, 10, 0, 1], [0, 5, 170, 32, 3], [0, 0, 17, 182, 29], [0, 1, 5, 43, 152]]},
    'D99': {'accuracy': 0.8761061946902655, 'macro_f1': 0.8706475966690842, 'balanced_accuracy': 0.8674517096721225, 'macro_auroc': 0.9849767655219335, 'class3_recall': 0.8464912280701754, 'class4_recall': 0.7661691542288557, 'confusion_matrix': [[284, 0, 0, 0, 2], [4, 190, 10, 0, 1], [0, 1, 169, 33, 7], [0, 0, 13, 193, 22], [0, 0, 6, 41, 154]]},
}
# v22's seed-42 H2 sweep for C and D (tau_s <- clamp(tau_s + delta, 0, 1)): the tau-only delay sweep must reproduce it.
V22_H2_SEED42 = {
    'C0.00': {'accuracy': 0.8938053097345132, 'macro_f1': 0.8900430576358016, 'macro_auroc': 0.986928283391767, 'class4_recall': 0.8258706467661692},
    'C0.05': {'accuracy': 0.8911504424778761, 'macro_f1': 0.8874654856769538, 'macro_auroc': 0.9870419407874493, 'class4_recall': 0.8407960199004975},
    'C0.10': {'accuracy': 0.8884955752212389, 'macro_f1': 0.8847611904799406, 'macro_auroc': 0.9866480951885397, 'class4_recall': 0.8407960199004975},
    'C0.20': {'accuracy': 0.8876106194690265, 'macro_f1': 0.8839827731944896, 'macro_auroc': 0.986160470199861, 'class4_recall': 0.8407960199004975},
    'C0.30': {'accuracy': 0.8840707964601769, 'macro_f1': 0.8801734347609445, 'macro_auroc': 0.9855716737536488, 'class4_recall': 0.8656716417910447},
    'D0.00': {'accuracy': 0.9097345132743363, 'macro_f1': 0.90640272771588, 'macro_auroc': 0.9890678208140686, 'class4_recall': 0.8706467661691543},
    'D0.05': {'accuracy': 0.9097345132743363, 'macro_f1': 0.90640272771588, 'macro_auroc': 0.9890678208140686, 'class4_recall': 0.8706467661691543},
    'D0.10': {'accuracy': 0.9097345132743363, 'macro_f1': 0.90640272771588, 'macro_auroc': 0.9890678208140686, 'class4_recall': 0.8706467661691543},
    'D0.20': {'accuracy': 0.9097345132743363, 'macro_f1': 0.90640272771588, 'macro_auroc': 0.9890678208140686, 'class4_recall': 0.8706467661691543},
    'D0.30': {'accuracy': 0.9097345132743363, 'macro_f1': 0.90640272771588, 'macro_auroc': 0.9890678208140686, 'class4_recall': 0.8706467661691543},
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


def set_all_seeds(seed: int) -> None:
    """Seed python `random`, numpy, torch CPU and every CUDA device; deterministic cuDNN."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


print(f"MISALIGNMENT ROBUSTNESS: conditions {EVAL_CONDITIONS} x seeds {SEEDS} = {len(RUN_ORDER)} models | arms {ARMS} | "
      f"delay {DELAYS} | jitter sigma {JITTER_SIGMAS} (one draw per test window, JITTER_SEED {JITTER_SEED})")
print(f"Results -> {RESULTS_CSV} | training/reuse log -> {TRAINING_CSV} | checkpoints -> {CHECKPOINT_DIR}")
print(f"Fixed configuration: audio strongest-{N_AUDIO_BINS}, Ta={TA_AUDIO}, sqrt, no window, no normalization "
      f"| sensor raw {N_SENSOR_CHANNELS} ch, Ts={TS_SENSOR} | LACA lag bias ON/OFF (beta {LACA_BETA}), {LACA_NUM_HEADS} heads, mlp g(Δτ) "
      f"{LAG_BIAS_PARAMS} params | Mamba {MAMBA_NUM_BLOCKS} x (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}) | "
      f"loss CE + {LAMBDA_ALIGN} InfoNCE + {LAMBDA_CONS} KL | {EPOCHS} epochs, batch {BATCH_SIZE}, AdamW {LEARNING_RATE}")

Device: cuda (Tesla T4)
MISALIGNMENT ROBUSTNESS: conditions ['C', 'D'] x seeds [42, 123, 2024, 7, 99] = 10 models | arms ['tau_only', 'content_shift'] | delay [0.0, 0.05, 0.1, 0.2, 0.3] | jitter sigma [0.0, 0.02, 0.05, 0.1, 0.2] (one draw per test window, JITTER_SEED 2025)
Results -> /kaggle/working/misalignment_robustness_results.csv | training/reuse log -> /kaggle/working/misalignment_robustness_training.csv | checkpoints -> /kaggle/working/v25_checkpoints
Fixed configuration: audio strongest-64, Ta=64, sqrt, no window, no normalization | sensor raw 6 ch, Ts=128 | LACA lag bias ON/OFF (beta 1.0), 4 heads, mlp g(Δτ) 97 params | Mamba 2 x (d_state 16, d_conv 2) | loss CE + 0.1 InfoNCE + 0.1 KL | 20 epochs, batch 32, AdamW 0.0001


In [2]:
# ==============================================================================
# mamba_ssm installation + fusion-backbone resolution.
# Order: an already-installed package, else a prebuilt wheel matched to this
# Python / torch / CUDA / C++ ABI (the reliable path on Kaggle), else a
# time-capped source build. If none succeeds, or there is no CUDA GPU, the
# pure-PyTorch fallback block is used and announced below.
# ==============================================================================
PIP_TIMEOUT_S = 240


def _env_tags():
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    try:
        cxx11_abi = bool(torch._C._GLIBCXX_USE_CXX11_ABI)
    except AttributeError:
        cxx11_abi = False
    return py_tag, torch_mm, torch.version.cuda, cxx11_abi


def _prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, cxx11_abi):
    """Best-matching .whl asset in `repo`'s GitHub releases (exact CUDA version first, then CUDA major)."""
    req = urllib.request.Request(f"https://api.github.com/repos/{repo}/releases",
                                 headers={"User-Agent": "kaggle-notebook", "Accept": "application/vnd.github+json"})
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            releases = json.load(resp)
    except Exception as exc:
        print(f"    could not query github.com/{repo}/releases: {exc!r}")
        return None
    abi = f"cxx11abi{'TRUE' if cxx11_abi else 'FALSE'}"
    cuda_exact = f"cu{cuda_ver.replace('.', '')}"
    cuda_major = f"cu{cuda_ver.split('.')[0]}"
    exact, major_only = [], []
    for rel in releases:
        for asset in rel.get("assets", []):
            name = asset.get("name", "")
            if not (name.endswith(".whl") and py_tag in name and abi in name and f"torch{torch_mm}" in name):
                continue
            if cuda_exact in name:
                exact.append(asset["browser_download_url"])
            elif cuda_major in name:
                major_only.append(asset["browser_download_url"])
    chosen = exact or major_only
    return chosen[0] if chosen else None


def _pip_install(args):
    """pip with a hard timeout and captured output, so a stuck build can never hang the kernel."""
    try:
        proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args,
                              timeout=PIP_TIMEOUT_S, capture_output=True, text=True, check=False)
    except subprocess.TimeoutExpired:
        print(f"    pip install {args} exceeded {PIP_TIMEOUT_S}s — killed")
        return False
    if proc.returncode != 0:
        print(f"    pip install {args} failed (exit {proc.returncode}): {(proc.stderr or proc.stdout)[-400:]}")
    return proc.returncode == 0


def install_mamba_ssm():
    if importlib.util.find_spec("mamba_ssm") is not None:
        return "already installed"
    tags = _env_tags()
    print(f"Installing mamba_ssm + causal-conv1d (python={tags[0]}, torch={tags[1]}, cuda={tags[2]}, cxx11abi={tags[3]})")
    urls = [_prebuilt_wheel_url(repo, *tags) for repo in ("Dao-AILab/causal-conv1d", "state-spaces/mamba")]
    if all(urls) and all(_pip_install([u, "--no-deps"]) for u in urls):
        return "prebuilt wheel"
    print("  no complete prebuilt-wheel match — trying a time-capped source build")
    if _pip_install(["causal-conv1d>=1.2.0", "--no-build-isolation"]) and _pip_install(["mamba-ssm", "--no-build-isolation"]):
        return "source build"
    return "failed"


MAMBA_CLASS = None
if torch.cuda.is_available():
    print(f"mamba_ssm install: {install_mamba_ssm()}")
    try:
        from mamba_ssm import Mamba as MAMBA_CLASS
    except ImportError as exc:
        print(f"mamba_ssm import failed: {exc!r}")

if MAMBA_CLASS is not None:
    FUSION_BACKBONE = "mamba_ssm.Mamba (real selective-scan SSM, CUDA)"
    print(f"✅ FUSION BACKBONE: {FUSION_BACKBONE}")
else:
    FUSION_BACKBONE = "FALLBACK pure-PyTorch gated-conv block (NOT a state-space model)"
    print("🚨" * 45)
    print(f"🚨  FUSION BACKBONE: {FUSION_BACKBONE}")
    print("🚨  " + ("no CUDA GPU detected" if not torch.cuda.is_available() else "mamba_ssm could not be installed/imported")
          + " — results will NOT match the reference numbers, which were produced with real mamba_ssm.")
    print("🚨" * 45)

Installing mamba_ssm + causal-conv1d (python=cp312, torch=2.10, cuda=12.8, cxx11abi=True)
    pip install ['https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/causal_conv1d-1.7.0%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_aarch64.whl', '--no-deps'] failed (exit 1): ERROR: causal_conv1d-1.7.0+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_aarch64.whl is not a supported wheel on this platform.

  no complete prebuilt-wheel match — trying a time-capped source build
mamba_ssm install: source build
✅ FUSION BACKBONE: mamba_ssm.Mamba (real selective-scan SSM, CUDA)


## 3. Data loading & preprocessing
Mission-level split, strongest-64 bin selection from training audio only, 2.56 s windows with real timestamps, and sensor standardization with training-split statistics. Leakage and timestamp checks are included.

In [3]:
# ==============================================================================
# DATA — TII UAV Realistic Fault Dataset: parse missions -> mission-level split
# -> strongest-64 bins from TRAINING audio only (frozen) -> 2.56 s windows with
# real per-window timestamps -> datasets (sensor standardized with TRAINING stats).
# ==============================================================================
if not DATASET_ROOT.exists():
    print(f"Cloning {DATASET_REPO_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO_URL, str(DATASET_BASE)], check=True)


def parse_mission(mission_dir: Path):
    """Real 6-axis IMU (seconds from mission start) + raw audio chunks with per-chunk timestamps.
    Audio chunk duration = the mission's IMU-measured duration / number of chunks (both come from the same
    continuous recording; the audio log carries no per-chunk timestamp)."""
    sensor_files = list(mission_dir.glob("*SensorCombined*.jsonl"))
    audio_files = list(mission_dir.glob("*AudioBuffer*.jsonl"))
    if not sensor_files or not audio_files:
        return None

    sensor_times, accel, gyro = [], [], []
    with open(sensor_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                t_sec = float(rec["timestamp"]) / 1e6
                sensor_times.append(t_sec)
                accel.append(rec.get("accelerometer_m_s2", [0.0, 0.0, 9.81]))
                gyro.append(rec.get("gyro_rad", [0.0, 0.0, 0.0]))
            except Exception:
                continue
    if len(sensor_times) < 50:
        return None
    sensor_times = np.array(sensor_times, dtype=np.float64)
    sensor_times = sensor_times - sensor_times[0]
    telemetry = np.concatenate([np.array(accel, dtype=np.float32), np.array(gyro, dtype=np.float32)], axis=-1)

    audio_chunks = []
    with open(audio_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                chunk = rec.get("audio_data", rec.get("buffer", rec.get("data", [])))
                if len(chunk) > 0:
                    audio_chunks.append(chunk)
            except Exception:
                continue
    if len(audio_chunks) < 10:
        return None
    audio_chunks = np.array(audio_chunks, dtype=np.float32)

    chunk_duration_s = (sensor_times[-1] - sensor_times[0]) / len(audio_chunks)
    audio_times = np.arange(len(audio_chunks), dtype=np.float64) * chunk_duration_s
    return {"sensor_times": sensor_times, "telemetry": telemetry, "audio_times": audio_times, "audio_chunks": audio_chunks}


def split_missions(dataset_root: Path):
    """Mission-level stratified split (70 / 15 / 15 per class) with its own fixed seed."""
    class_missions = defaultdict(list)
    for class_dir in sorted(dataset_root.iterdir()):
        if not class_dir.is_dir() or class_dir.name.startswith("."):
            continue
        try:
            label = int(class_dir.name)
        except ValueError:
            continue
        for mission_dir in sorted(class_dir.iterdir()):
            if mission_dir.is_dir() and not mission_dir.name.startswith("."):
                class_missions[label].append(mission_dir)

    random.seed(SPLIT_SEED)
    splits = {"train": {}, "val": {}, "test": {}}
    for label in sorted(class_missions):
        missions = class_missions[label]
        random.shuffle(missions)
        n_train = max(1, int(len(missions) * TRAIN_FRACTION))
        n_val = max(1, int(len(missions) * VAL_FRACTION))
        splits["train"][label] = missions[:n_train]
        splits["val"][label] = missions[n_train:n_train + n_val]
        splits["test"][label] = missions[n_train + n_val:]
        print(f"Class {label} ({len(missions)} flights) -> train {n_train}, val {len(splits['val'][label])}, "
              f"test {len(splits['test'][label])}")
    return splits


def select_strongest_bins(train_missions, n_bins):
    """Indices of the n_bins rFFT magnitude bins with the highest mean magnitude over TRAINING audio only
    (unsupervised energy, no labels). Returned sorted; frozen and reused for val/test."""
    magnitude_sum, n_chunks, fft_len = None, 0, None
    for mission in train_missions:
        mag = np.abs(np.fft.rfft(mission["audio_chunks"], axis=1))
        fft_len = mag.shape[1] if fft_len is None else min(fft_len, mag.shape[1])
        mag = mag[:, :fft_len]
        if magnitude_sum is not None and len(magnitude_sum) > fft_len:
            magnitude_sum = magnitude_sum[:fft_len]
        magnitude_sum = mag.sum(axis=0) if magnitude_sum is None else magnitude_sum + mag.sum(axis=0)
        n_chunks += mag.shape[0]
    mean_magnitude = magnitude_sum / n_chunks
    return np.sort(np.argsort(mean_magnitude)[::-1][:n_bins]).astype(int)


def audio_features(audio_chunks, bin_indices):
    """raw chunk -> rFFT (no window) -> |.| -> frozen strongest bins -> sqrt (no normalization)."""
    mag = np.abs(np.fft.rfft(audio_chunks, axis=1))
    valid = bin_indices[bin_indices < mag.shape[1]]
    feats = np.zeros((mag.shape[0], len(bin_indices)), dtype=np.float32)
    feats[:, :len(valid)] = np.sqrt(mag[:, valid]).astype(np.float32)
    return feats


def slice_windows(mission, label, bin_indices):
    """2.56 s windows (1.28 s step). Sensor: TS_SENSOR real IMU samples chosen evenly by index inside the window;
    audio: TA_AUDIO real chunk timestamps chosen the same way (a window with fewer than 2 audio chunks — the
    recording edge — falls back to an even grid). tau_a / tau_s = those REAL timestamps relative to window start."""
    s_times, telemetry = mission["sensor_times"], mission["telemetry"]
    a_times = mission["audio_times"]
    a_feats = audio_features(mission["audio_chunks"], bin_indices)
    max_time = min(s_times[-1], a_times[-1])

    windows, t_start = [], 0.0
    while t_start + WINDOW_SEC <= max_time:
        t_end = t_start + WINDOW_SEC
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < MIN_SENSOR_SAMPLES_PER_WINDOW:
            t_start += STEP_SEC
            continue

        s_idx = np.where(s_mask)[0]
        tau_s_abs = s_times[s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]]
        x_s = np.zeros((TS_SENSOR, N_SENSOR_CHANNELS), dtype=np.float32)
        for c in range(N_SENSOR_CHANNELS):
            x_s[:, c] = np.interp(tau_s_abs, s_times[s_mask], telemetry[s_mask, c])

        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx = np.where(a_mask)[0]
            tau_a_abs = a_times[a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]]
        else:
            tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
        x_a = np.zeros((TA_AUDIO, len(bin_indices)), dtype=np.float32)
        for c in range(len(bin_indices)):
            x_a[:, c] = np.interp(tau_a_abs, a_times, a_feats[:, c], left=0.0, right=0.0)

        windows.append({
            "x_a": x_a,
            "x_s": x_s,
            "label": int(label),
            "tau_a": (tau_a_abs - t_start).astype(np.float32),
            "tau_s": (tau_s_abs - t_start).astype(np.float32),
            "window_sec": np.float32(WINDOW_SEC),
        })
        t_start += STEP_SEC
    return windows


class UAVWindowDataset(Dataset):
    """Windows -> tensors. Sensor channels are standardized with mean/std that must come from the TRAINING
    split (computed here only when none are passed, i.e. for the training set). tau is normalized by the
    window length."""

    def __init__(self, windows, sensor_mean=None, sensor_std=None):
        self.windows = windows
        if sensor_mean is None or sensor_std is None:
            all_s = np.concatenate([w["x_s"] for w in windows], axis=0)
            sensor_mean = all_s.mean(axis=0, keepdims=True)
            sensor_std = all_s.std(axis=0, keepdims=True) + SENSOR_STD_EPS
        self.sensor_mean, self.sensor_std = sensor_mean, sensor_std

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        window_sec = float(w["window_sec"])
        return {
            "x_a": torch.tensor(w["x_a"], dtype=torch.float32),
            "x_s": torch.tensor((w["x_s"] - self.sensor_mean) / self.sensor_std, dtype=torch.float32),
            "tau_a": torch.tensor(w["tau_a"], dtype=torch.float32) / window_sec,
            "tau_s": torch.tensor(w["tau_s"], dtype=torch.float32) / window_sec,
            "label": torch.tensor(w["label"], dtype=torch.long),
        }


def split_fingerprint(windows):
    """(n windows, sum of raw sensor values, sum of sensor timestamps, class counts) — deterministic data check."""
    x_s = np.stack([w["x_s"] for w in windows]).astype(np.float64)
    tau_s = np.stack([w["tau_s"] for w in windows]).astype(np.float64)
    counts = tuple(np.bincount([w["label"] for w in windows], minlength=N_CLASSES).tolist())
    return (len(windows), round(float(x_s.sum()), 3), round(float(tau_s.sum()), 3), counts)


# --- 1. split + parse ---
mission_splits = split_missions(DATASET_ROOT)
split_dirs = {s: [m for label in sorted(mission_splits[s]) for m in mission_splits[s][label]] for s in mission_splits}
assert not set(split_dirs["train"]) & (set(split_dirs["val"]) | set(split_dirs["test"])), "LEAKAGE: mission in two splits"
parsed_missions = {m: parse_mission(m) for s in split_dirs for m in tqdm(split_dirs[s], desc=f"parse {s}", leave=False)}

# --- 2. strongest-64 bins from TRAINING audio only ---
bin_source_missions = [m for m in split_dirs["train"] if parsed_missions[m] is not None]
assert set(bin_source_missions) <= set(split_dirs["train"])
SELECTED_BINS = select_strongest_bins([parsed_missions[m] for m in bin_source_missions], N_AUDIO_BINS)
assert len(SELECTED_BINS) == N_AUDIO_BINS
print(f"[LEAKAGE CHECK] strongest-{N_AUDIO_BINS} bins computed from {len(bin_source_missions)} TRAINING missions only "
      f"(0 of {len(split_dirs['val']) + len(split_dirs['test'])} val/test missions) -> frozen for all splits: "
      f"{SELECTED_BINS.tolist()}")

# --- 3. windows (the same frozen bins for every split) ---
split_windows = {}
for split in ("train", "val", "test"):
    split_windows[split] = []
    for label in sorted(mission_splits[split]):
        for m in mission_splits[split][label]:
            if parsed_missions[m] is not None:
                split_windows[split].extend(slice_windows(parsed_missions[m], label, SELECTED_BINS))
    print(f"{split:<5}: {len(split_windows[split])} windows | audio {TA_AUDIO}x{N_AUDIO_BINS}, sensor {TS_SENSOR}x{N_SENSOR_CHANNELS}")
DATA_FINGERPRINT = {s: split_fingerprint(split_windows[s]) for s in split_windows}

# --- 4. datasets: sensor mean/std from the TRAINING split only, reused for val/test ---
train_ds = UAVWindowDataset(split_windows["train"])
val_ds = UAVWindowDataset(split_windows["val"], train_ds.sensor_mean, train_ds.sensor_std)
test_ds = UAVWindowDataset(split_windows["test"], train_ds.sensor_mean, train_ds.sensor_std)
_train_s = np.concatenate([w["x_s"] for w in split_windows["train"]], axis=0)
_val_s = np.concatenate([w["x_s"] for w in split_windows["val"]], axis=0)
assert val_ds.sensor_mean is train_ds.sensor_mean and test_ds.sensor_std is train_ds.sensor_std
assert np.array_equal(train_ds.sensor_mean, _train_s.mean(axis=0, keepdims=True))
print(f"[LEAKAGE CHECK] sensor standardization stats = TRAINING-split stats (same objects reused for val/test) | "
      f"train mean {np.round(train_ds.sensor_mean.ravel(), 3).tolist()} vs val's own mean "
      f"{np.round(_val_s.mean(axis=0), 3).tolist()} (not used)")
del _train_s, _val_s

# --- 5. real timestamps reach the model: tau must differ across windows ---
_tau_a = np.stack([train_ds[i]["tau_a"].numpy() for i in range(len(train_ds))])
_tau_s = np.stack([train_ds[i]["tau_s"].numpy() for i in range(len(train_ds))])
_n_unique_a, _n_unique_s = len(np.unique(_tau_a, axis=0)), len(np.unique(_tau_s, axis=0))
assert _n_unique_a > 1 and _n_unique_s > 1, "real timestamps are NOT reaching the model (tau identical for every window)"
assert not np.allclose(_tau_s, np.linspace(0.0, 1.0, TS_SENSOR)[None, :]), "tau_s equals a synthetic linspace grid"
print(f"[TIMESTAMP CHECK] {len(train_ds)} training windows: {_n_unique_a} distinct tau_a rows, {_n_unique_s} distinct "
      f"tau_s rows -> real per-window timestamps reach HTT/LACA ✅")
del _tau_a, _tau_s

Cloning https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git ...


Cloning into '/kaggle/working/tii_uav_dataset'...


Class 0 (19 flights) -> train 13, val 2, test 4
Class 1 (20 flights) -> train 14, val 3, test 3
Class 2 (20 flights) -> train 14, val 3, test 3
Class 3 (20 flights) -> train 14, val 3, test 3
Class 4 (20 flights) -> train 14, val 3, test 3


Updating files: 100% (217/217), done.


parse train:   0%|          | 0/69 [00:00<?, ?it/s]

parse val:   0%|          | 0/14 [00:00<?, ?it/s]

parse test:   0%|          | 0/16 [00:00<?, ?it/s]

[LEAKAGE CHECK] strongest-64 bins computed from 69 TRAINING missions only (0 of 30 val/test missions) -> frozen for all splits: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766]
train: 4934 windows | audio 64x64, sensor 128x6
val  : 954 windows | audio 64x64, sensor 128x6
test : 1130 windows | audio 64x64, sensor 128x6
[LEAKAGE CHECK] sensor standardization stats = TRAINING-split stats (same objects reused for val/test) | train mean [0.05299999937415123, -0.0430000014603138, -9.78600025177002, 0.004000000189989805, 0.003000000026077032, -0.0] vs val's own mean [0.029999999329447746, -0.061000000685453415, -9.748000144958496, 0.004999999888241291, 0.003000000026077032, -0.0] (not used)
[TIMESTAMP CHECK] 4934 training windows: 4934 distinct 

## 4. Model definition — two explicit switches
- **`HeterogeneousTemporalTokenization(d_model, time_enabled)`.** When ON: `norm(z + e_m + time_embedding(tau))`, which is v18 unchanged. When OFF: `norm(z + e_m)`. `time_embedding` is still built, so the RNG draws are the same, but it is never called.
- **`AeroSyncMamba(htt_time, laca_lag)`.** Sets the LACA scale to β = 1.0 (ON) or exactly 0.0 (OFF), which is the v18 β = 0 code path.

The v18 LayerNorm and zero-init of g are kept for the ON conditions. The warm-up and `set_beta` are removed: β never changes during a run. The cell checks that every condition has the same parameter count and prints where each switch acts.

In [4]:
# ==============================================================================
# MODEL — encoders -> HTT (modality + continuous-time embedding of the REAL tau)
# -> bi-directional LACA (lag-aware cross-attention; lag bias beta * LN_grid(g(Δτ)), beta = 1.0 when
#    LACA_LAG_BIAS_ENABLED else 0.0; 4 heads). v22: HTT's time embedding is switchable too.
# -> [H_a ; SEP ; H_s] -> 2 Mamba blocks -> attention pooling -> classifier.
# Auxiliary per-modality heads feed only the consistency loss.
# ==============================================================================
class ContinuousTimeEmbedding(nn.Module):
    def __init__(self, d_model: int, num_frequencies: int = TIME_EMBED_FREQUENCIES):
        super().__init__()
        self.frequencies = nn.Parameter(
            torch.exp(torch.linspace(0, math.log(1000.0), num_frequencies)), requires_grad=False
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * num_frequencies, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        if tau.dim() == 2:
            tau = tau.unsqueeze(-1)
        angles = tau * self.frequencies * 2.0 * math.pi
        return self.mlp(torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1))


class HeterogeneousTemporalTokenization(nn.Module):
    """token = LayerNorm(features + modality embedding [+ time embedding(tau)]); modality 0 = audio, 1 = sensor.
    v22: time_enabled = HTT_TIME_EMBED_ENABLED. When False the time embedding is skipped entirely (no temporal
    position information enters through HTT); the module is still built so every condition draws the same RNG."""

    def __init__(self, d_model: int, time_enabled: bool = True):
        super().__init__()
        self.d_model = d_model
        self.time_enabled = time_enabled
        self.modality_embeddings = nn.Embedding(2, d_model)
        self.time_embedding = ContinuousTimeEmbedding(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z: torch.Tensor, tau: torch.Tensor, modality_idx: int) -> torch.Tensor:
        mod_idx = torch.tensor(modality_idx, device=z.device, dtype=torch.long)
        e_m = self.modality_embeddings(mod_idx).view(1, 1, self.d_model)
        if not self.time_enabled:
            return self.norm(z + e_m)                       # HTT_TIME_EMBED_ENABLED = False
        return self.norm(z + e_m + self.time_embedding(tau))


class GridLayerNorm(nn.Module):
    """LayerNorm for the lag-bias grid (v18).

    g(Δτ) produces ONE scalar per (query i, key j) pair, so its output is a (B, T_m, T_n) grid with no feature
    dimension > 1. A LayerNorm over a size-1 dimension would subtract each value from itself and return 0 for every
    input. Instead, the normalization is taken over each window's whole (T_m, T_n) grid, i.e. every bias value that
    enters that window's attention matrix. It is computed separately for each window: every window has its own real
    timestamps, and mixing windows would make one window's bias depend on the rest of the batch and differ between
    training and evaluation. There are no affine parameters, so beta is the only source of scale.

    Consequences: the subtracted mean is one constant per window, and softmax is unchanged by a constant added to
    every score, so only the division by the std affects attention. For an all-zero grid (the zero-init state) the
    output is exactly 0. As g develops variance v, the output std grows smoothly as sqrt(v / (v + eps)) toward 1."""

    def __init__(self, eps: float = LAG_BIAS_LN_EPS):
        super().__init__()
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:   # x: (B, T_m, T_n)
        return F.layer_norm(x, x.shape[-2:], eps=self.eps)


class PairwiseLACA(nn.Module):
    """Cross-attention from modality m to n with a learned lag bias g(Δτ) added to the scores (scaled by β)."""

    def __init__(self, d_model: int, num_heads: int = LACA_NUM_HEADS, beta: float = LACA_BETA):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.beta = beta
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.lag_bias = nn.Sequential(nn.Linear(1, LAG_BIAS_HIDDEN), nn.Tanh(), nn.Linear(LAG_BIAS_HIDDEN, 1))
        # v18 ZERO-INIT: g's final Linear starts at exactly 0 -> g(Δτ) = 0 for every Δτ and every seed. The layer was
        # already built (same RNG draws as v16), and nn.init.zeros_ draws no random numbers, so every other
        # layer's initial weights and the training RNG stream are unchanged.
        nn.init.zeros_(self.lag_bias[-1].weight)
        nn.init.zeros_(self.lag_bias[-1].bias)
        self.lag_bias_norm = GridLayerNorm()   # v18: no parameters

    def lag_bias_term(self, tau_m, tau_n):
        """LN_grid(g(Δτ)), BEFORE beta. Returns (B, 1, T_m, T_n), broadcast over the heads."""
        delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)   # (B, T_m, T_n, 1)
        g = self.lag_bias(delta_tau).squeeze(-1)                                # (B, T_m, T_n): one scalar per (i, j)
        return self.lag_bias_norm(g).unsqueeze(1)                               # normalized per window -> (B, 1, T_m, T_n)

    def forward(self, H_m, H_n, tau_m, tau_n):
        B, T_m, _ = H_m.shape
        _, T_n, _ = H_n.shape
        Q = self.w_q(H_m).view(B, T_m, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        lag_bias = self.lag_bias_term(tau_m, tau_n)

        attn = F.softmax(scores + self.beta * lag_bias, dim=-1)   # beta = 1.0 (lag bias ON) or exactly 0.0 (OFF)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T_m, self.d_model)

        attn_mean = attn.mean(dim=1)
        tau_diff = tau_n.unsqueeze(1) - tau_m.unsqueeze(2)
        expected_lag = (attn_mean * tau_diff).sum(dim=(1, 2)) / float(T_m)
        return self.out_proj(out), expected_lag


class BiModalLACAFusion(nn.Module):
    def __init__(self, d_model: int, beta: float = LACA_BETA):
        super().__init__()
        self.laca_as = PairwiseLACA(d_model, beta=beta)   # beta scales the lag bias in BOTH directions
        self.laca_sa = PairwiseLACA(d_model, beta=beta)
        self.gamma_as = nn.Parameter(torch.tensor(0.5))
        self.gamma_sa = nn.Parameter(torch.tensor(0.5))

    def forward(self, Ha, Hs, tau_a, tau_s):
        H_a_from_s, lag_as = self.laca_as(Ha, Hs, tau_a, tau_s)
        H_s_from_a, lag_sa = self.laca_sa(Hs, Ha, tau_s, tau_a)
        return Ha + self.gamma_as * H_a_from_s, Hs + self.gamma_sa * H_s_from_a, {"lag_as": lag_as, "lag_sa": lag_sa}


class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM."""

    def __init__(self, d_model: int, d_state: int, d_conv: int, expand: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.mamba(self.norm(x))


class FallbackGatedConvBlock(nn.Module):
    """CPU-only stand-in when mamba_ssm is unavailable: depthwise causal conv + gated MLP. NOT a state-space
    model (no hidden state, no A/B/C, no discretization)."""

    def __init__(self, d_model: int, d_conv: int, expand: int):
        super().__init__()
        d_inner = expand * d_model
        self.in_proj = nn.Linear(d_model, d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(d_inner, d_inner, kernel_size=d_conv, padding=d_conv - 1, groups=d_inner, bias=True)
        self.mixer = nn.Sequential(nn.Linear(d_inner, d_inner), nn.GELU(), nn.Linear(d_inner, d_inner))
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        L = x.shape[1]
        x_branch, z_branch = self.in_proj(self.norm(x)).chunk(2, dim=-1)
        x_conv = F.silu(self.conv1d(x_branch.transpose(1, 2))[:, :, :L].transpose(1, 2))
        return x + self.out_proj(self.mixer(x_conv) * F.silu(z_branch))


def build_fusion_blocks(d_model, num_blocks, d_state, d_conv, expand=MAMBA_EXPAND):
    if MAMBA_CLASS is not None:
        return nn.ModuleList([MambaBlock(d_model, d_state, d_conv, expand) for _ in range(num_blocks)])
    return nn.ModuleList([FallbackGatedConvBlock(d_model, d_conv, expand) for _ in range(num_blocks)])


# The reference results were produced by a model whose constructor drew, from the torch RNG, the initial weights
# of a 4-block (d_state 16, d_conv 4) Mamba stack before the final 2-block stack was built from the SAME starting
# RNG state. Every module built afterwards (pooling, classifier, auxiliary heads) and the whole training RNG stream
# (shuffle order, dropout masks) depend on the RNG state that 4-block draw leaves behind. To reproduce those
# results exactly, the constructor performs the same draw and discards it; the discarded blocks are never part of
# the model. This is a fixed step of weight initialization, not a configuration option.
RNG_REFERENCE_STACK = (4, 16, 4)   # (num_blocks, d_state, d_conv) whose initialization is drawn and discarded


class AeroSyncMamba(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_classes: int = N_CLASSES, htt_time: bool = True, laca_lag: bool = False):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes
        self.htt_time_enabled = htt_time                  # HTT_TIME_EMBED_ENABLED
        self.laca_lag_enabled = laca_lag                  # LACA_LAG_BIAS_ENABLED
        self.laca_beta = LACA_BETA if laca_lag else 0.0   # OFF = exactly 0.0 (v18's beta = 0 code path)

        self.sensor_encoder = nn.Sequential(
            nn.Conv1d(N_SENSOR_CHANNELS, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=3, padding=1), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        self.audio_encoder = nn.Sequential(
            nn.Linear(N_AUDIO_BINS, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )
        self.htt = HeterogeneousTemporalTokenization(d_model, time_enabled=htt_time)
        self.laca = BiModalLACAFusion(d_model, beta=self.laca_beta)   # plain float: no params, no RNG draws
        self.sep_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        rng_before = torch.get_rng_state()
        build_fusion_blocks(d_model, *RNG_REFERENCE_STACK)
        rng_after_reference = torch.get_rng_state()
        torch.set_rng_state(rng_before)
        self.mamba_blocks = build_fusion_blocks(d_model, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        torch.set_rng_state(rng_after_reference)

        self.pool_attn = nn.Sequential(nn.Linear(d_model, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )
        self.aux_a = nn.Linear(d_model, n_classes)
        self.aux_s = nn.Linear(d_model, n_classes)

    def forward(self, xa, xs, ta, ts):
        B = xa.shape[0]
        za = self.audio_encoder(xa)
        zs = self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)

        Ha = self.htt(za, ta, modality_idx=0)
        Hs = self.htt(zs, ts, modality_idx=1)
        H_tilde_a, H_tilde_s, lags = self.laca(Ha, Hs, ta, ts)
        za_p = H_tilde_a.mean(dim=1)
        zs_p = H_tilde_s.mean(dim=1)

        F_seq = torch.cat([H_tilde_a, self.sep_token.expand(B, -1, -1), H_tilde_s], dim=1)
        for block in self.mamba_blocks:
            F_seq = block(F_seq)

        weights = F.softmax(self.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = self.classifier(zf)

        pooled = {"z_a": za_p, "z_s": zs_p}
        aux_logits = {"y_a": self.aux_a(za_p), "y_s": self.aux_s(zs_p)}
        return logits, pooled, aux_logits, lags


def verify_architecture(model):
    """Finalized architecture, read from the constructed modules, including both v22 switches."""
    assert model.htt.time_enabled == model.htt_time_enabled
    beta = LACA_BETA if model.laca_lag_enabled else 0.0
    for name in ("laca_as", "laca_sa"):
        laca = getattr(model.laca, name)
        assert laca.num_heads == LACA_NUM_HEADS, f"{name}: {laca.num_heads} heads"
        assert laca.beta == beta, f"{name}: beta {laca.beta} != {beta} for LACA_LAG_BIAS_ENABLED={model.laca_lag_enabled}"
        assert sum(p.numel() for p in laca.lag_bias.parameters()) == LAG_BIAS_PARAMS
        assert isinstance(laca.lag_bias_norm, GridLayerNorm) and not list(laca.lag_bias_norm.parameters())
    assert len(model.mamba_blocks) == MAMBA_NUM_BLOCKS
    mamba_params = sum(p.numel() for p in model.mamba_blocks.parameters())
    if MAMBA_CLASS is not None:
        for block in model.mamba_blocks:
            m = block.mamba
            assert (m.d_state, m.d_conv, m.expand) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND)
        assert mamba_params == EXPECTED_MAMBA_PARAMS, f"Mamba params {mamba_params:,} != {EXPECTED_MAMBA_PARAMS:,}"
    return mamba_params


_probe_counts = {}
for _c, (_h, _l) in CONDITIONS.items():
    _probe = AeroSyncMamba(htt_time=_h, laca_lag=_l)
    verify_architecture(_probe)
    _probe_counts[_c] = sum(p.numel() for p in _probe.parameters())
    del _probe
assert len(set(_probe_counts.values())) == 1, f"parameter count depends on the condition: {_probe_counts}"
_probe = AeroSyncMamba()
print(f"Model: {_probe_counts['A']:,} params in every condition {list(CONDITIONS)} (both modules always built) | fusion "
      f"blocks: {type(_probe.mamba_blocks[0]).__name__} x {len(_probe.mamba_blocks)} ({verify_architecture(_probe):,} "
      f"params) | LACA {LACA_NUM_HEADS} heads, g(Δτ) {LAG_BIAS_PARAMS} params | backbone: {FUSION_BACKBONE}")
del _probe
print("[SWITCHES] HTT_TIME_EMBED_ENABLED -> HeterogeneousTemporalTokenization.forward: ON norm(z + e_m + time_embedding(tau)) "
      "(v18, unchanged) | OFF norm(z + e_m)")
print(f"[SWITCHES] LACA_LAG_BIAS_ENABLED -> PairwiseLACA.forward: softmax(scores + beta * LN_grid(g(Δτ))) with beta = "
      f"{LACA_BETA} (ON) or exactly 0.0 (OFF, v18's beta = 0 path)")
print("[REUSE PRECONDITION] v18's HTT had no switch: its forward always added time_embedding(tau) -> v18 at beta = 0.0 "
      "IS Condition A (HTT-time ON, LACA-lag OFF).")


# --- LayerNorm placement: trace real tensor shapes through one LACA direction on 2 training windows ---
_ta = torch.stack([train_ds[i]["tau_a"] for i in range(2)])
_ts = torch.stack([train_ds[i]["tau_s"] for i in range(2)])
_probe = AeroSyncMamba(htt_time=True, laca_lag=True)
with torch.no_grad():
    for _name, _laca, _tm, _tn in (("laca_as", _probe.laca.laca_as, _ta, _ts), ("laca_sa", _probe.laca.laca_sa, _ts, _ta)):
        _dt = (_tm.unsqueeze(2) - _tn.unsqueeze(1)).unsqueeze(-1)
        _g = _laca.lag_bias(_dt).squeeze(-1)
        _out = _laca.lag_bias_term(_tm, _tn)
        print(f"[LAYERNORM] {_name}: Δτ {tuple(_dt.shape)} -> g(Δτ) {tuple(_g.shape)} = (B, T_m, T_n) -> "
              f"LN over dims {tuple(_g.shape[-2:])} ({_g.shape[-2] * _g.shape[-1]:,} values per window, per window) -> "
              f"{tuple(_out.shape)} -> x beta -> added to scores (B, {LACA_NUM_HEADS} heads, T_m, T_n)")
        assert float(_g.abs().max()) == 0.0 and float(_out.abs().max()) == 0.0, "zero-init g should give an all-zero term"
del _probe, _ta, _ts
print("[LAYERNORM] placement: PairwiseLACA.forward -> scores + beta * LN_grid(g(Δτ)); the LayerNorm is applied to g's "
      "scalar output BEFORE beta.")
print("[LAYERNORM] why this shape: g outputs one scalar per (i, j) pair, so there is no feature dim > 1 (LN over a "
      "size-1 dim would output 0 for every input). Normalizing over each window's full (T_m, T_n) grid covers every "
      "bias value in that window's attention matrix. It is per window, never across the batch, so a window's bias "
      "does not depend on its batch-mates and is the same in train and eval. No affine: beta is the only scale. The "
      "subtracted mean is one constant per window, which softmax ignores; only the std normalization affects "
      "attention.")

Model: 1,688,788 params in every condition ['A', 'B', 'C', 'D'] (both modules always built) | fusion blocks: MambaBlock x 2 (874,496 params) | LACA 4 heads, g(Δτ) 97 params | backbone: mamba_ssm.Mamba (real selective-scan SSM, CUDA)
[SWITCHES] HTT_TIME_EMBED_ENABLED -> HeterogeneousTemporalTokenization.forward: ON norm(z + e_m + time_embedding(tau)) (v18, unchanged) | OFF norm(z + e_m)
[SWITCHES] LACA_LAG_BIAS_ENABLED -> PairwiseLACA.forward: softmax(scores + beta * LN_grid(g(Δτ))) with beta = 1.0 (ON) or exactly 0.0 (OFF, v18's beta = 0 path)
[REUSE PRECONDITION] v18's HTT had no switch: its forward always added time_embedding(tau) -> v18 at beta = 0.0 IS Condition A (HTT-time ON, LACA-lag OFF).
[LAYERNORM] laca_as: Δτ (2, 64, 128, 1) -> g(Δτ) (2, 64, 128) = (B, T_m, T_n) -> LN over dims (64, 128) (8,192 values per window, per window) -> (2, 1, 64, 128) -> x beta -> added to scores (B, 4 heads, T_m, T_n)
[LAYERNORM] laca_sa: Δτ (2, 128, 64, 1) -> g(Δτ) (2, 128, 64) = (B, T_m, T_n) -> 

## 5. Metrics, switch checks, H2 sweep and the training function
**Evaluation.** `evaluate_probs` returns the 4 standard metrics, the full confusion matrix, Class 3 and Class 4 recall, and the 3→4 and 4→3 counts.

**Switch checks** (on a fixed real batch, before training and at the best checkpoint):
- `htt_time_contribution` measures the time embedding's actual effect: max|HTT(z, τ) − norm(z + e_m)| and max|HTT(z, τ) − HTT(z, τ + 0.3)|. Both must be **exactly 0** when HTT time is OFF, and above 0 when ON.
- `lag_bias_contribution` (from v18) measures max|β · LN(g(Δτ))|. It is asserted **exactly 0 at every epoch** when LACA lag is OFF. When ON it starts at 0, because g is zero-initialized, and must be above 0 by the end.

**`h2_sweep`** is the v4/v5 H2 procedure: sensor timestamps become clamp(τ_s + δ, 0, 1) at the model input, and macro-F1 retention is reported relative to δ = 0.

**`train_one_run(seed, condition)`:**
1. Resets every RNG and builds the model with the condition's switches.
2. Checks that its initial weights and RNG state equal the v18 run for that seed. They are the same in all 4 conditions.
3. Trains, then evaluates the best checkpoint.

In [5]:
# ==============================================================================
# TRAINING — loss (unchanged), metrics (4 standard + confusion matrix + Class 3/4
# recall + 3->4 / 4->3), switch checks (HTT time term, LACA lag term), the H2
# temporal-shift sweep, and train_one_run(seed, condition).
# ==============================================================================
class AeroSyncLoss(nn.Module):
    """L = CE_classweighted(logits, y) + LAMBDA_ALIGN * InfoNCE(z_a, z_s) + LAMBDA_CONS * KL-consistency."""

    def __init__(self, class_weights):
        super().__init__()
        self.lambda_align = LAMBDA_ALIGN
        self.lambda_cons = LAMBDA_CONS
        self.temperature = INFONCE_TEMPERATURE
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def _infonce(self, z1, z2):
        sim = torch.matmul(F.normalize(z1, p=2, dim=-1), F.normalize(z2, p=2, dim=-1).t()) / self.temperature
        labels = torch.arange(z1.shape[0], device=z1.device)
        return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))

    def forward(self, logits, targets, pooled, aux_logits):
        loss_cls = self.ce(logits, targets)
        loss_align = self._infonce(pooled["z_a"], pooled["z_s"])
        p_fused = F.softmax(logits.detach(), dim=-1)
        loss_cons = (
            F.kl_div(F.log_softmax(aux_logits["y_a"], dim=-1), p_fused, reduction="batchmean") +
            F.kl_div(F.log_softmax(aux_logits["y_s"], dim=-1), p_fused, reduction="batchmean")
        ) / 2.0
        total = loss_cls + self.lambda_align * loss_align + self.lambda_cons * loss_cons
        return total, {"loss_cls": loss_cls.item(), "loss_align": loss_align.item(), "loss_cons": loss_cons.item()}


def compute_metrics(y_true, y_prob):
    """accuracy, macro-F1, balanced accuracy, macro one-vs-rest AUROC from class probabilities (prediction = argmax)."""
    y_pred = y_prob.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_auroc": float(roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")),
    }


def train_class_weights(windows):
    """Inverse class frequency over the TRAINING windows."""
    counts = np.bincount([w["label"] for w in windows], minlength=N_CLASSES)
    return torch.tensor(counts.sum() / (len(counts) * np.maximum(counts, 1)), dtype=torch.float32)


def _batch_to_device(batch, non_blocking=False):
    return tuple(batch[k].to(device, non_blocking=non_blocking) for k in ("x_a", "x_s", "tau_a", "tau_s", "label"))


def predict_probs(model, loader):
    """Softmax probabilities and true labels over a loader (eval mode, no grad), in loader order."""
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            xa, xs, ta, ts, y = _batch_to_device(batch)
            logits, _, _, _ = model(xa, xs, ta, ts)
            probs.append(F.softmax(logits, dim=-1).cpu().numpy())
            labels.append(y.cpu().numpy())
    return np.concatenate(probs), np.concatenate(labels)


train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS)
CLASS_WEIGHTS = train_class_weights(split_windows["train"]).to(device)
TEST_LABELS = np.array([w["label"] for w in split_windows["test"]], dtype=np.int64)   # test-window order, fixed


# Fixed probe batch for the lag-contribution check: timestamps of the first BATCH_SIZE training windows, read by
# index (no DataLoader, no RNG). The lag-bias term depends only on the timestamps.
LAG_PROBE_TAUS = tuple(torch.stack([train_ds[i][k] for i in range(min(BATCH_SIZE, len(train_ds)))])
                       for k in ("tau_a", "tau_s"))


def lag_bias_contribution(model):
    """Per LACA direction: max|beta * LN_grid(g(Δτ))|, the exact term PairwiseLACA adds to the attention scores
    (computed by the same lag_bias_term method, with the beta currently in effect), plus the raw std of g before
    the LayerNorm. g has no dropout or BatchNorm: this uses no random numbers and changes no state."""
    ta, ts = (t.to(device) for t in LAG_PROBE_TAUS)
    out = {}
    with torch.no_grad():
        for name, laca, tau_m, tau_n in (("laca_as", model.laca.laca_as, ta, ts), ("laca_sa", model.laca.laca_sa, ts, ta)):
            term = laca.lag_bias_term(tau_m, tau_n)
            contrib = laca.beta * term
            g_raw = laca.lag_bias((tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)).squeeze(-1)
            out[name] = {"beta": laca.beta, "contrib_absmax": float(contrib.abs().max()),
                         "contrib_finite": bool(torch.isfinite(contrib).all()), "g_raw_std": float(g_raw.std())}
    return out


def g_final_layer_absmax(model):
    """max|weight| and max|bias| of g's final Linear(32, 1), per LACA direction."""
    return {name: (float(getattr(model.laca, name).lag_bias[-1].weight.detach().abs().max()),
                   float(getattr(model.laca, name).lag_bias[-1].bias.detach().abs().max()))
            for name in ("laca_as", "laca_sa")}


from sklearn.metrics import confusion_matrix


def evaluate_probs(y_true, y_prob):
    """4 standard metrics + full 5x5 confusion matrix + per-class recall + Class 3/4 recall + 3->4 / 4->3 counts."""
    cm = confusion_matrix(y_true, y_prob.argmax(axis=1), labels=list(range(N_CLASSES)))
    recall = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)
    return {**compute_metrics(y_true, y_prob), "confusion_matrix": cm,
            **{f"recall_c{c}": float(recall[c]) for c in range(N_CLASSES)},
            "class3_recall": float(recall[3]), "class4_recall": float(recall[4]),
            "err_3to4": int(cm[3, 4]), "err_4to3": int(cm[4, 3])}


def evaluate_cm(cm, metrics):
    """Same fields from a stored confusion matrix (reused runs); the 4 metrics come from the stored values."""
    cm = np.asarray(cm)
    recall = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)
    return {**{m: metrics[m] for m in METRICS}, "confusion_matrix": cm,
            **{f"recall_c{c}": float(recall[c]) for c in range(N_CLASSES)},
            "class3_recall": float(recall[3]), "class4_recall": float(recall[4]),
            "err_3to4": int(cm[3, 4]), "err_4to3": int(cm[4, 3])}


def print_confusion(ev, tag):
    cm = ev["confusion_matrix"]
    print(f"[CONFUSION {tag}] rows = TRUE severity, cols = PREDICTED severity (test set, {int(cm.sum())} windows)")
    print("            " + "".join(f"pred {j:>2} " for j in range(N_CLASSES)) + "|  recall")
    for i in range(N_CLASSES):
        print(f"   true {i:>2} " + "".join(f"{int(cm[i, j]):>8}" for j in range(N_CLASSES)) + f"  |  {ev[f'recall_c{i}']:.4f}")
    print(f"[CLASS 3/4 {tag}] Class 3 recall {ev['class3_recall']:.4f} | Class 4 recall {ev['class4_recall']:.4f} | "
          f"3->4 (true 3, pred 4) {ev['err_3to4']} | 4->3 (true 4, pred 3) {ev['err_4to3']}")


METRICS = ["accuracy", "macro_f1", "balanced_accuracy", "macro_auroc"]
SWITCH_PROBE_IDX = np.linspace(0, len(train_ds) - 1, BATCH_SIZE).astype(int)   # fixed real batch, read by index


def htt_time_contribution(model):
    """What HTT's time embedding actually contributes, on a fixed real batch (eval mode, RNG untouched):
    term = max|HTT(z, tau) - norm(z + e_m)| and tau_sens = max|HTT(z, tau) - HTT(z, tau + 0.3)|, per modality."""
    items = [train_ds[int(i)] for i in SWITCH_PROBE_IDX]
    xa, xs, ta, ts = (torch.stack([it[k] for it in items]).to(device) for k in ("x_a", "x_s", "tau_a", "tau_s"))
    was_training = model.training
    cuda_devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
    out = {}
    with torch.random.fork_rng(devices=cuda_devices), torch.no_grad():
        model.eval()
        za = model.audio_encoder(xa)
        zs = model.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)
        for name, z, tau, idx in (("audio", za, ta, 0), ("sensor", zs, ts, 1)):
            e_m = model.htt.modality_embeddings(torch.tensor(idx, device=device)).view(1, 1, model.d_model)
            actual = model.htt(z, tau, modality_idx=idx)
            out[name] = {"term": float((actual - model.htt.norm(z + e_m)).abs().max()),
                         "tau_sens": float((actual - model.htt(z, tau + 0.3, modality_idx=idx)).abs().max())}
    model.train(was_training)
    return out


def check_switches(model, tag, when, final=False):
    """Print and enforce both switches: a disabled contribution must be exactly 0."""
    h = htt_time_contribution(model)
    lag = lag_bias_contribution(model)
    lag_max = max(r["contrib_absmax"] for r in lag.values())
    h_max = max(max(v["term"], v["tau_sens"]) for v in h.values())
    print(f"{tag} [SWITCH CHECK {when}] HTT_TIME_EMBED_ENABLED={model.htt.time_enabled}: time-embedding term max "
          + ", ".join(f"{k} {v['term']:.3e}" for k, v in h.items()) + " | output change for tau+0.3: "
          + ", ".join(f"{k} {v['tau_sens']:.3e}" for k, v in h.items()))
    print(f"{tag} [SWITCH CHECK {when}] LACA_LAG_BIAS_ENABLED={model.laca_lag_enabled} (beta read back: "
          f"{model.laca.laca_as.beta} / {model.laca.laca_sa.beta}): max|beta * LN(g(Δτ))| = {lag_max!r}")
    if not model.htt_time_enabled:
        assert h_max == 0.0, f"HTT time embedding is OFF but contributes {h_max!r}"
        print(f"{tag} [SWITCH CHECK {when}] ✅ HTT time OFF: its contribution is EXACTLY 0 and HTT ignores tau entirely")
    else:
        assert h_max > 0.0, "HTT time embedding is ON but contributes nothing"
        print(f"{tag} [SWITCH CHECK {when}] ✅ HTT time ON: the time embedding changes the tokens and responds to tau")
    if not model.laca_lag_enabled:
        assert all(r["contrib_finite"] and r["contrib_absmax"] == 0.0 for r in lag.values()), f"LACA lag OFF but {lag}"
        print(f"{tag} [SWITCH CHECK {when}] ✅ LACA lag OFF: the lag term added to the attention scores is EXACTLY 0")
    elif final and lag_max == 0.0:
        print(f"🚨 {tag} LACA lag is ON but its contribution is still 0 after training — the switch is not working")
    elif lag_max == 0.0:
        print(f"{tag} [SWITCH CHECK {when}] LACA lag ON (beta {LACA_BETA}); the term is 0 only because g's final layer is "
              f"zero-initialized (v18 form) — it grows from the first update")
    else:
        print(f"{tag} [SWITCH CHECK {when}] ✅ LACA lag ON: the lag term is active ({lag_max:.4f})")
    return h, lag_max


def predict_probs_shift(model, loader, delta):
    """H2 (v4/v5): sensor timestamps -> clamp(tau_s + delta, 0, 1) at the model input (HTT and LACA both see it)."""
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            xa, xs, ta, ts, y = _batch_to_device(batch)
            logits = model(xa, xs, ta, torch.clamp(ts + delta, 0.0, 1.0))[0]
            probs.append(F.softmax(logits, dim=-1).cpu().numpy())
            labels.append(y.cpu().numpy())
    return np.concatenate(probs), np.concatenate(labels)


def h2_sweep(model, cond, seed, clean_ev):
    """Macro-F1 retention under artificial audio <-> IMU delays; returns one row per offset."""
    tag = f"[H2 {cond} seed={seed}]"
    rows, base_f1, base_probs = [], None, None
    print(f"{tag} temporal-shift robustness: tau_s <- clamp(tau_s + delta, 0, 1)")
    print(f"      {'offset':>8} {'accuracy':>9} {'macro-F1':>9} {'AUROC':>8} {'C4 recall':>10} {'retention':>10}  predictions")
    for delta in H2_OFFSETS:
        probs, labels = predict_probs_shift(model, test_loader, delta)
        ev = evaluate_probs(labels, probs)
        if delta == 0.0:
            base_f1, base_probs = ev["macro_f1"], probs
            clean_match = max(abs(ev[m] - clean_ev[m]) for m in METRICS)
        same = np.array_equal(probs, base_probs)
        rows.append({"condition": cond, "seed": seed, "offset": delta, "accuracy": ev["accuracy"], "macro_f1": ev["macro_f1"],
                     "macro_auroc": ev["macro_auroc"], "class4_recall": ev["class4_recall"],
                     "retention_pct": 100.0 * ev["macro_f1"] / base_f1, "identical_to_offset0": same})
        print(f"      {'+' + format(delta, '.2f'):>8} {ev['accuracy']:>9.4f} {ev['macro_f1']:>9.4f} {ev['macro_auroc']:>8.4f} "
              f"{ev['class4_recall']:>10.4f} {rows[-1]['retention_pct']:>9.2f}%  {'identical to +0.00' if same else 'changed'}")
    print(f"{tag} offset 0 (with the clamp) vs the clean test evaluation: max |delta| {clean_match:.1e}"
          + (" ✅" if clean_match < 1e-9 else " ⚠️ the clamp alters some timestamps even at offset 0"))
    if not CONDITIONS[cond][0] and not CONDITIONS[cond][1]:
        flat = all(r["identical_to_offset0"] for r in rows)
        print(f"{tag} Condition D uses no timestamps anywhere -> the curve must be exactly flat: "
              + ("✅ every offset gives bit-identical predictions" if flat else "🚨 NOT flat — a timestamp path is still active"))
    return rows


def train_one_run(seed: int, cond: str):
    """Train one model of `cond` with `seed` applied to every RNG, evaluate it on the test set.
    Returns (trained model on CPU, result dict incl. test probabilities and confusion matrix)."""
    t0 = time.time()
    htt_on, laca_on = CONDITIONS[cond]
    tag = f"[{cond} seed={seed} HTT-time {'ON' if htt_on else 'OFF'} / LACA-lag {'ON' if laca_on else 'OFF'}]"
    set_all_seeds(seed)
    model = AeroSyncMamba(htt_time=htt_on, laca_lag=laca_on).to(device)
    init_sha1 = hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes()
                                      for v in model.state_dict().values())).hexdigest()
    rng_sha1 = hashlib.sha1(torch.get_rng_state().numpy().tobytes()).hexdigest()
    mamba_params = verify_architecture(model)
    ref = V18_BETA0_RUNS[seed]
    paired_ok = init_sha1[:12] == ref["init_sha1"] and rng_sha1[:12] == ref["rng_sha1"]
    print(f"{tag} HTT_TIME_EMBED_ENABLED = {model.htt.time_enabled} | LACA_LAG_BIAS_ENABLED = {model.laca_lag_enabled} "
          f"(beta {model.laca.laca_as.beta}) — read from the constructed model | fusion {type(model.mamba_blocks[0]).__name__} "
          f"({mamba_params:,} params)")
    print(f"{tag} [PAIRING] initial weights {init_sha1[:12]} vs v18 {ref['init_sha1']} | RNG {rng_sha1[:12]} vs v18 "
          f"{ref['rng_sha1']} -> " + ("✅ identical start and RNG stream (same in all 4 conditions)" if paired_ok else
                                      "🚨 MISMATCH — this seed's conditions are not cleanly paired"))
    check_switches(model, tag, "before training")

    criterion = AeroSyncLoss(CLASS_WEIGHTS)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)

    best_val_f1, best_state, best_epoch = 0.0, None, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = defaultdict(float)
        for batch in tqdm(train_loader, desc=f"{cond} seed {seed} epoch {epoch:02d}/{EPOCHS}", leave=False):
            xa, xs, ta, ts, y = _batch_to_device(batch, non_blocking=True)
            optimizer.zero_grad()
            logits, pooled, aux_logits, _ = model(xa, xs, ta, ts)
            loss, parts = criterion(logits, y, pooled, aux_logits)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            running["loss"] += loss.item()
            for k, v in parts.items():
                running[k] += v
        scheduler.step()

        lag = lag_bias_contribution(model)
        lag_max = max(r["contrib_absmax"] for r in lag.values())
        if not laca_on:
            assert all(r["contrib_finite"] and r["contrib_absmax"] == 0.0 for r in lag.values()), f"lag leak at epoch {epoch}"

        val_probs, val_labels = predict_probs(model, val_loader)
        val_f1 = compute_metrics(val_labels, val_probs)["macro_f1"]
        improved = val_f1 > best_val_f1
        if improved:
            best_val_f1, best_state, best_epoch = val_f1, copy.deepcopy(model.state_dict()), epoch
        n = len(train_loader)
        print(f"{tag} epoch {epoch:02d}/{EPOCHS} | lag term {lag_max:.4f}{' (exactly 0 ✅)' if not laca_on else ''} | train "
              f"loss {running['loss'] / n:.4f} (cls {running['loss_cls'] / n:.4f}, align {running['loss_align'] / n:.4f}, "
              f"cons {running['loss_cons'] / n:.4f}) | val macro-F1 {val_f1:.4f}{'  * best' if improved else ''}")

    model.load_state_dict(best_state)
    h_final, lag_final = check_switches(model, tag, f"best checkpoint, epoch {best_epoch}", final=True)
    test_probs, test_labels = predict_probs(model, test_loader)
    ev = evaluate_probs(test_labels, test_probs)
    run_name = f"{cond}_seed{seed}"
    torch.save(best_state, CHECKPOINT_DIR / f"{run_name}_best.pt")
    np.save(PROBS_DIR / f"{run_name}_test_probs.npy", test_probs)
    np.save(PROBS_DIR / f"{run_name}_test_labels.npy", test_labels)

    result = {
        "condition": cond, "seed": seed, "htt_time": htt_on, "laca_lag": laca_on,
        **ev,
        "best_val_macro_f1": round(best_val_f1, 6),
        "best_epoch": best_epoch,
        "n_test_windows": len(test_labels),
        "init_sha1": init_sha1[:12],
        "rng_sha1": rng_sha1[:12],
        "paired_with_v18": paired_ok,
        "htt_time_term_final": max(v["term"] for v in h_final.values()),
        "lag_term_final": lag_final,
        "train_seconds": round(time.time() - t0, 1),
        "test_probs": test_probs,
        "test_labels": test_labels,
    }
    return model, result

## 6. Misalignment injection (evaluation only) and its self-checks
- **Clean test tensors.** v22's `test_ds` is stacked once, in loader order. The code checks that τ_s already lies in
  [0, 1], so the δ = 0 clamp changes nothing.
- **Jitter noise.** One vector z of standard normals, one per test window, drawn from its own generator seeded with
  `JITTER_SEED`. Each σ uses σ·z. The sha1 of every σ's vector is printed.
- **Arm 1 (τ-only).** τ_s is shifted and clamped. Audio, IMU and τ_a are untouched.
- **Arm 2 (content shift).** The IMU window is re-sliced from the shifted time, using the same code as v18's
  `slice_windows`, and τ_s is set to the samples' true times relative to the audio window. The checks:
  - at shift 0, every re-sliced window must equal v18's test window bit for bit;
  - the fixed evaluation subset is printed, along with how many windows were dropped and why.
- `eval_inputs` runs the model in v22's test-loader order with batches of 32 and no DataLoader, so it draws no random
  numbers.

In [6]:
# ==============================================================================
# MISALIGNMENT INJECTION — Arm 1 tau-only (H2 mechanism), Arm 2 content shift (real misalignment, honest timestamps)
# ==============================================================================
def stack_dataset(ds):
    """All windows of a UAVWindowDataset as stacked tensors, in index order (= v22's unshuffled test_loader order)."""
    items = [ds[i] for i in range(len(ds))]
    return {k: torch.stack([it[k] for it in items]) for k in ("x_a", "x_s", "tau_a", "tau_s", "label")}


CLEAN = stack_dataset(test_ds)
N_TEST = len(test_ds)
assert np.array_equal(CLEAN["label"].numpy(), TEST_LABELS)
assert torch.equal(torch.clamp(CLEAN["tau_s"], 0.0, 1.0), CLEAN["tau_s"]), "clean tau_s outside [0, 1]"
print(f"[CLEAN] {N_TEST} test windows stacked in test-loader order | tau_s in [{float(CLEAN['tau_s'].min()):.4f}, "
      f"{float(CLEAN['tau_s'].max()):.4f}] -> clamp(tau_s + 0, 0, 1) == tau_s ✅")

# --- jitter: ONE standard-normal draw per test window, own Generator, reused for every sigma / arm / model ---
JITTER_Z = np.random.default_rng(JITTER_SEED).standard_normal(N_TEST)
JITTER_SHIFTS = {s: s * JITTER_Z for s in JITTER_SIGMAS if s > 0}


def _sha(a):
    return hashlib.sha1(np.ascontiguousarray(a, dtype=np.float64).tobytes()).hexdigest()[:12]


JITTER_SHA1 = {s: _sha(v) for s, v in JITTER_SHIFTS.items()}
print(f"[JITTER] seed {JITTER_SEED} (numpy Generator, independent of every training seed) | one draw per test window "
      f"({N_TEST}) | z sha1 {_sha(JITTER_Z)} | z mean {JITTER_Z.mean():+.3f}, SD {JITTER_Z.std():.3f}")
for s, v in JITTER_SHIFTS.items():
    print(f"[JITTER] sigma {s:.2f}: per-window shift sha1 {JITTER_SHA1[s]} | |shift| mean {np.abs(v).mean():.4f}, max "
          f"{np.abs(v).max():.4f} window fractions ({np.abs(v).max() * WINDOW_SEC:.3f} s)")

PERTURBATIONS = [("delay", d) for d in DELAYS if d > 0] + [("jitter", s) for s in JITTER_SIGMAS if s > 0]


def shift_vector(kind, mag):
    """Per-test-window shift in window fractions."""
    if kind == "none" or mag == 0:
        return np.zeros(N_TEST)
    return np.full(N_TEST, float(mag)) if kind == "delay" else JITTER_SHIFTS[mag]


def shift_sha1(kind, mag):
    return JITTER_SHA1[mag] if kind == "jitter" else f"const {mag}"


# --- ARM 1: tau-only — clamp(tau_s + shift, 0, 1); nothing else changes ---
def tau_only_inputs(kind, mag):
    if kind == "delay":   # exactly v22's H2: python-float delta added to the float32 tau_s, then clamped
        return {**CLEAN, "tau_s": torch.clamp(CLEAN["tau_s"] + mag, 0.0, 1.0)}
    shift = torch.tensor(shift_vector(kind, mag), dtype=torch.float32).view(-1, 1)
    return {**CLEAN, "tau_s": torch.clamp(CLEAN["tau_s"] + shift, 0.0, 1.0)}


# --- ARM 2: content shift — re-slice each test window's IMU from the shifted time, honest timestamps ---
def _window_grid(mission):
    """Start times of the windows slice_windows returns for this mission (same grid, accumulation and skip rule)."""
    s_times, a_times = mission["sensor_times"], mission["audio_times"]
    max_time = min(s_times[-1], a_times[-1])
    starts, t_start = [], 0.0
    while t_start + WINDOW_SEC <= max_time:
        if (((s_times >= t_start) & (s_times <= t_start + WINDOW_SEC)).sum()) >= MIN_SENSOR_SAMPLES_PER_WINDOW:
            starts.append(t_start)
        t_start += STEP_SEC
    return starts


def _shift_valid(mission, t_start, shift_s):
    """The shifted IMU window [t_start + shift_s, t_start + WINDOW_SEC + shift_s] lies inside the flight and has enough samples."""
    s_times = mission["sensor_times"]
    lo, hi = t_start + shift_s, t_start + WINDOW_SEC + shift_s
    return lo >= 0.0 and hi <= s_times[-1] and int(((s_times >= lo) & (s_times <= hi)).sum()) >= MIN_SENSOR_SAMPLES_PER_WINDOW


def _window_content_shift(mission, a_feats, label, t_start, shift_s):
    """slice_windows' body for ONE window, with the IMU window moved by shift_s seconds. Audio is exactly v18's (window
    [t_start, t_end]). tau_s = the selected IMU samples' TRUE times relative to the AUDIO window start (not clamped).
    With shift_s = 0.0 this is v18's window, bit for bit (checked below)."""
    s_times, telemetry = mission["sensor_times"], mission["telemetry"]
    a_times = mission["audio_times"]
    t_end = t_start + WINDOW_SEC
    s_mask = (s_times >= t_start + shift_s) & (s_times <= t_end + shift_s)
    s_idx = np.where(s_mask)[0]
    tau_s_abs = s_times[s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]]
    x_s = np.zeros((TS_SENSOR, N_SENSOR_CHANNELS), dtype=np.float32)
    for c in range(N_SENSOR_CHANNELS):
        x_s[:, c] = np.interp(tau_s_abs, s_times[s_mask], telemetry[s_mask, c])

    a_mask = (a_times >= t_start) & (a_times <= t_end)
    if a_mask.sum() >= 2:
        a_idx = np.where(a_mask)[0]
        tau_a_abs = a_times[a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]]
    else:
        tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
    x_a = np.zeros((TA_AUDIO, len(SELECTED_BINS)), dtype=np.float32)
    for c in range(len(SELECTED_BINS)):
        x_a[:, c] = np.interp(tau_a_abs, a_times, a_feats[:, c], left=0.0, right=0.0)
    return {"x_a": x_a, "x_s": x_s, "label": int(label), "tau_a": (tau_a_abs - t_start).astype(np.float32),
            "tau_s": (tau_s_abs - t_start).astype(np.float32), "window_sec": np.float32(WINDOW_SEC)}


# index of every clean test window: (mission, label, t_start), in split_windows["test"] order
TEST_INDEX, _A_FEATS = [], {}
for label in sorted(mission_splits["test"]):
    for m in mission_splits["test"][label]:
        if parsed_missions[m] is not None:
            _A_FEATS[m] = audio_features(parsed_missions[m]["audio_chunks"], SELECTED_BINS)
            TEST_INDEX.extend((m, label, t) for t in _window_grid(parsed_missions[m]))
assert len(TEST_INDEX) == N_TEST, f"{len(TEST_INDEX)} indexed test windows vs {N_TEST}"

# fixed Arm-2 subset: windows whose shifted IMU window is valid at EVERY perturbation
_invalid_by = defaultdict(int)
CONTENT_MASK = np.ones(N_TEST, dtype=bool)
for kind, mag in PERTURBATIONS:
    sv = shift_vector(kind, mag)
    for i, (m, label, t) in enumerate(TEST_INDEX):
        if CONTENT_MASK[i] and not _shift_valid(parsed_missions[m], t, float(sv[i]) * WINDOW_SEC):
            CONTENT_MASK[i] = False
            _invalid_by[f"{kind} {mag}"] += 1
CONTENT_IDX = np.where(CONTENT_MASK)[0]
print(f"[CONTENT SHIFT] evaluation subset: {len(CONTENT_IDX)} of {N_TEST} test windows (dropped {N_TEST - len(CONTENT_IDX)} "
      f"whose shifted IMU window would leave the flight; first failing perturbation: {dict(_invalid_by)}) | per class: "
      f"{np.bincount(TEST_LABELS[CONTENT_IDX], minlength=N_CLASSES).tolist()} (full test set "
      f"{np.bincount(TEST_LABELS, minlength=N_CLASSES).tolist()})")


def content_inputs(kind, mag):
    sv = shift_vector(kind, mag)
    wins = [_window_content_shift(parsed_missions[TEST_INDEX[i][0]], _A_FEATS[TEST_INDEX[i][0]], TEST_INDEX[i][1],
                                  TEST_INDEX[i][2], float(sv[i]) * WINDOW_SEC) for i in CONTENT_IDX]
    return stack_dataset(UAVWindowDataset(wins, train_ds.sensor_mean, train_ds.sensor_std))


CONTENT_INPUTS = {("none", 0.0): content_inputs("none", 0.0)}
_clean_sub = {k: v[CONTENT_IDX] for k, v in CLEAN.items()}
CONTENT_BASELINE_EXACT = all(torch.equal(CONTENT_INPUTS[("none", 0.0)][k], _clean_sub[k]) for k in _clean_sub)
assert CONTENT_BASELINE_EXACT, "content-shift re-slicer at shift 0 does NOT reproduce v18's test windows — injection bug"
print(f"[CONTENT SHIFT] shift 0: all {len(CONTENT_IDX)} re-sliced windows equal v18's test windows bit for bit "
      f"(x_a, x_s, tau_a, tau_s, label) ✅")
for kind, mag in PERTURBATIONS:
    CONTENT_INPUTS[(kind, mag)] = content_inputs(kind, mag)
    ts = CONTENT_INPUTS[(kind, mag)]["tau_s"]
    assert torch.equal(CONTENT_INPUTS[(kind, mag)]["x_a"], _clean_sub["x_a"]), "content shift changed the audio"
    print(f"[CONTENT SHIFT] {kind:<6} {mag:.2f}: IMU content moved by {'%.3f s' % (mag * WINDOW_SEC) if kind == 'delay' else 'sigma*z*W per window'} "
          f"| audio unchanged ✅ | honest tau_s range [{float(ts.min()):+.3f}, {float(ts.max()):+.3f}] (unclamped) | "
          f"x_s changed in {int((CONTENT_INPUTS[(kind, mag)]['x_s'] != _clean_sub['x_s']).flatten(1).any(1).sum())} windows")
del _clean_sub


def eval_inputs(model, inputs):
    """Softmax probabilities in input order: batches of BATCH_SIZE in v22's test-loader order, eval mode, no RNG."""
    model.eval()
    n, probs = len(inputs["label"]), []
    with torch.no_grad():
        for i in range(0, n, BATCH_SIZE):
            xa, xs, ta, ts = (inputs[k][i:i + BATCH_SIZE].to(device) for k in ("x_a", "x_s", "tau_a", "tau_s"))
            probs.append(F.softmax(model(xa, xs, ta, ts)[0], dim=-1).cpu().numpy())
    return np.concatenate(probs)


def ev5(labels, probs):
    ev = evaluate_probs(labels, probs)
    return {**{m: ev[m] for m in METRICS}, "class3_recall": ev["class3_recall"], "class4_recall": ev["class4_recall"],
            "confusion_matrix": ev["confusion_matrix"]}


[CLEAN] 1130 test windows stacked in test-loader order | tau_s in [0.0000, 1.0000] -> clamp(tau_s + 0, 0, 1) == tau_s ✅
[JITTER] seed 2025 (numpy Generator, independent of every training seed) | one draw per test window (1130) | z sha1 e11717ed88ec | z mean -0.012, SD 1.002
[JITTER] sigma 0.02: per-window shift sha1 f1cb9969e16c | |shift| mean 0.0159, max 0.0755 window fractions (0.193 s)
[JITTER] sigma 0.05: per-window shift sha1 6699fc9f8d02 | |shift| mean 0.0399, max 0.1888 window fractions (0.483 s)
[JITTER] sigma 0.10: per-window shift sha1 79bae0987fe9 | |shift| mean 0.0797, max 0.3775 window fractions (0.966 s)
[JITTER] sigma 0.20: per-window shift sha1 1dd60fc1841b | |shift| mean 0.1595, max 0.7551 window fractions (1.933 s)
[CONTENT SHIFT] evaluation subset: 1115 of 1130 test windows (dropped 15 whose shifted IMU window would leave the flight; first failing perturbation: {'delay 0.1': 1, 'delay 0.2': 3, 'delay 0.3': 4, 'jitter 0.02': 7}) | per class: [279, 203, 209, 226, 198] 

## 7. Driver — Part 1 (reuse or retrain) and Part 2 (all misalignment evaluations), model by model
For each seed, first C and then D:
1. **Obtain the model.** A found checkpoint is loaded and must reproduce v22's clean result. Otherwise the run is
   retrained with v22's `train_one_run` and checked against v22. Either way, one row goes to the training CSV.
2. **Clean evaluation.** The model is scored on the clean test set. The result is compared with the v22 test loader's
   output, which must be bit-identical.
3. **Arm 1.** 4 delays and 4 jitter levels. For D, each result must be bit-identical to its clean result. For
   seed 42, the delays must reproduce v22's H2 sweep.
4. **Arm 2.** Its baseline plus 4 delays and 4 jitter levels.

Every evaluation is written to the results CSV as soon as it finishes. The model is then freed and GPU memory released.

In [7]:
# ==============================================================================
# DRIVER — per model: obtain (reuse v22 checkpoint or retrain) -> clean -> Arm 1 -> Arm 2; CSV after every evaluation
# ==============================================================================
import glob

for f in (RESULTS_CSV, TRAINING_CSV):
    if f.exists():
        f.rename(f.with_suffix(f".{datetime.now():%Y%m%d_%H%M%S}.bak.csv"))
RESULTS_FIELDS = ["arm", "perturbation", "magnitude", "condition", "seed", "model_source", "n_windows"] + METRICS + \
                 ["class3_recall", "class4_recall", "identical_to_baseline", "jitter_seed", "shift_sha1",
                  "confusion_matrix", "finished_at"]
TRAINING_FIELDS = ["condition", "seed", "source", "checkpoint"] + METRICS + ["class3_recall", "class4_recall",
                   "confusion_matrix", "v22_max_abs_delta", "v22_confusion_identical", "reproduces_v22", "best_epoch",
                   "best_val_macro_f1", "init_sha1", "rng_sha1", "paired_with_v18", "train_seconds", "finished_at"]


def _append(path, fields, row):
    write_header = not path.exists()
    with open(path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        if write_header:
            w.writeheader()
        w.writerow({k: row.get(k) for k in fields})


def _find_checkpoint(name):
    for pattern in CHECKPOINT_GLOBS:
        hits = sorted(glob.glob(pattern.format(name=name), recursive=True))
        if hits:
            return hits[0]
    return None


def _vs_v22(cond, seed, ev):
    ref = V22_CD_RUNS[f"{cond}{seed}"]
    delta = max(abs(ev[m] - ref[m]) for m in METRICS)
    cm_same = bool(np.array_equal(np.asarray(ev["confusion_matrix"]), np.array(ref["confusion_matrix"])))
    return delta, cm_same, delta <= SAME_SEED_TOLERANCE and cm_same


def obtain_model(seed, cond):
    """Reuse a checkpoint that reproduces v22's clean result; otherwise retrain with v22's train_one_run."""
    htt_on, laca_on = CONDITIONS[cond]
    tag = f"[{cond} seed={seed}]"
    path = _find_checkpoint(f"{cond}_seed{seed}_best.pt")
    if path is not None:
        model = AeroSyncMamba(htt_time=htt_on, laca_lag=laca_on).to(device)
        model.load_state_dict(torch.load(path, map_location=device))
        verify_architecture(model)
        ev = ev5(TEST_LABELS, eval_inputs(model, CLEAN))
        delta, cm_same, ok = _vs_v22(cond, seed, ev)
        print(f"{tag} checkpoint {path}: clean macro-F1 {ev['macro_f1']:.4f} vs v22 {V22_CD_RUNS[f'{cond}{seed}']['macro_f1']:.4f} "
              f"| max |delta| {delta:.1e} | confusion matrix {'identical' if cm_same else 'DIFFERENT'}")
        if ok:
            print(f"{tag} ✅ REUSED checkpoint (reproduces v22) — not retrained")
            _append(TRAINING_CSV, TRAINING_FIELDS, {"condition": cond, "seed": seed, "source": "reused checkpoint",
                    "checkpoint": path, **ev, "confusion_matrix": json.dumps(np.asarray(ev["confusion_matrix"]).tolist()),
                    "v22_max_abs_delta": delta, "v22_confusion_identical": cm_same, "reproduces_v22": ok,
                    "finished_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")})
            return model, "reused checkpoint", (delta, cm_same, ok)
        print(f"{tag} 🚨 checkpoint does NOT reproduce v22 -> retraining instead")
        del model
    else:
        print(f"{tag} no checkpoint found ({', '.join(p.format(name=f'{cond}_seed{seed}_best.pt') for p in CHECKPOINT_GLOBS[:2])}, "
              f"...) -> RETRAINING with v22's train_one_run")
    model, result = train_one_run(seed, cond)
    delta, cm_same, ok = _vs_v22(cond, seed, result)
    ref = V22_CD_RUNS[f"{cond}{seed}"]
    print(f"{tag} [REPRODUCTION vs v22] " + " | ".join(f"{m} {result[m]:.4f} (v22 {ref[m]:.4f}, {result[m] - ref[m]:+.4f})"
                                                      for m in METRICS)
          + f" | confusion matrix {'identical' if cm_same else 'DIFFERENT'} -> "
          + ("✅ reproduces v22" if ok else f"🚨 does NOT reproduce v22 (max |delta| {delta:.4f})"))
    _append(TRAINING_CSV, TRAINING_FIELDS, {**{k: result.get(k) for k in TRAINING_FIELDS}, "condition": cond, "seed": seed,
            "source": "retrained in v25", "checkpoint": str(CHECKPOINT_DIR / f"{cond}_seed{seed}_best.pt"),
            "confusion_matrix": json.dumps(np.asarray(result["confusion_matrix"]).tolist()),
            "v22_max_abs_delta": delta, "v22_confusion_identical": cm_same, "reproduces_v22": ok,
            "finished_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")})
    return model, "retrained in v25", (delta, cm_same, ok)


RESULTS = []                 # every evaluation (also in RESULTS_CSV)
MODEL_SOURCES, REPRO = {}, {}
TAU_INVARIANCE = {}          # (seed, kind, mag) -> D bit-identical to its clean result under Arm 1
H2_REPRO = {}                # (cond, delta) -> max |metric - v22 H2| for seed 42
LOADER_MATCH = {}            # (cond, seed) -> eval_inputs(CLEAN) bit-identical to v22's predict_probs(test_loader)


def log_eval(arm, kind, mag, cond, seed, labels, probs, base_probs, sha):
    ev = ev5(labels, probs)
    row = {"arm": arm, "perturbation": kind, "magnitude": mag, "condition": cond, "seed": seed,
           "model_source": MODEL_SOURCES[(cond, seed)], "n_windows": len(labels), **ev,
           "identical_to_baseline": bool(base_probs is not None and np.array_equal(probs, base_probs)),
           "jitter_seed": JITTER_SEED if kind == "jitter" else None, "shift_sha1": sha,
           "confusion_matrix": json.dumps(np.asarray(ev["confusion_matrix"]).tolist()),
           "finished_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
    RESULTS.append(row)
    _append(RESULTS_CSV, RESULTS_FIELDS, row)
    return row


t_driver, n_trained = time.time(), 0
for slot, (seed, cond) in enumerate(RUN_ORDER, start=1):
    print("\n" + "=" * 100 + f"\nMODEL {slot}/{len(RUN_ORDER)} — Condition {cond} (HTT-time {'ON' if CONDITIONS[cond][0] else 'OFF'}, "
          f"LACA-lag {'ON' if CONDITIONS[cond][1] else 'OFF'}), seed {seed}\n" + "=" * 100)
    model, source, REPRO[(cond, seed)] = obtain_model(seed, cond)
    MODEL_SOURCES[(cond, seed)] = source
    n_trained += source.startswith("retrained")
    model.eval()

    # --- clean (shared zero-misalignment baseline of Arm 1) ---
    clean_probs = eval_inputs(model, CLEAN)
    loader_probs, _ = predict_probs(model, test_loader)
    LOADER_MATCH[(cond, seed)] = bool(np.array_equal(clean_probs, loader_probs))
    base = log_eval("tau_only", "none", 0.0, cond, seed, TEST_LABELS, clean_probs, clean_probs, "none")
    print(f"[{cond} seed={seed}] clean: macro-F1 {base['macro_f1']:.4f} | Class 4 recall {base['class4_recall']:.4f} | "
          f"eval path == v22 test_loader: {'bit-identical ✅' if LOADER_MATCH[(cond, seed)] else '⚠️ differs'}")

    # --- Arm 1: tau-only ---
    for kind, mag in PERTURBATIONS:
        probs = eval_inputs(model, tau_only_inputs(kind, mag))
        row = log_eval("tau_only", kind, mag, cond, seed, TEST_LABELS, probs, clean_probs, shift_sha1(kind, mag))
        if cond == "D":
            TAU_INVARIANCE[(seed, kind, mag)] = row["identical_to_baseline"]
            if not row["identical_to_baseline"]:
                print(f"🚨🚨🚨 [{cond} seed={seed}] tau-only {kind} {mag}: Condition D's predictions CHANGED — D reads no "
                      f"timestamps, so this is a BUG in the injection / switch code, not a finding 🚨🚨🚨")
        if seed == 42 and kind == "delay":
            ref = V22_H2_SEED42[f"{cond}{mag:.2f}"]
            H2_REPRO[(cond, mag)] = max(abs(row[k] - ref[k]) for k in ("accuracy", "macro_f1", "macro_auroc", "class4_recall"))
    print(f"[{cond} seed={seed}] Arm 1 tau-only macro-F1: " + " | ".join(
        f"{r['perturbation']} {r['magnitude']}: {r['macro_f1']:.4f}{' (=)' if r['identical_to_baseline'] else ''}"
        for r in RESULTS if r["arm"] == "tau_only" and r["condition"] == cond and r["seed"] == seed))

    # --- Arm 2: content shift (fixed subset, own baseline) ---
    sub_labels = TEST_LABELS[CONTENT_IDX]
    c_base = eval_inputs(model, CONTENT_INPUTS[("none", 0.0)])
    log_eval("content_shift", "none", 0.0, cond, seed, sub_labels, c_base, c_base, "none")
    for kind, mag in PERTURBATIONS:
        log_eval("content_shift", kind, mag, cond, seed, sub_labels, eval_inputs(model, CONTENT_INPUTS[(kind, mag)]),
                 c_base, shift_sha1(kind, mag))
    print(f"[{cond} seed={seed}] Arm 2 content-shift macro-F1: " + " | ".join(
        f"{r['perturbation']} {r['magnitude']}: {r['macro_f1']:.4f}"
        for r in RESULTS if r["arm"] == "content_shift" and r["condition"] == cond and r["seed"] == seed))

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elapsed = time.time() - t_driver
    print(f">>> {slot}/{len(RUN_ORDER)} models done ({source}) | {n_trained} retrained so far | elapsed {elapsed / 60:.1f} min")

print(f"\nAll {len(RUN_ORDER)} models evaluated: {n_trained} retrained, {len(RUN_ORDER) - n_trained} reused, "
      f"{len(RESULTS)} evaluations, in {(time.time() - t_driver) / 60:.1f} min. Results: {RESULTS_CSV}")



MODEL 1/10 — Condition C (HTT-time ON, LACA-lag ON), seed 42
[C seed=42] no checkpoint found (/kaggle/input/**/v22_checkpoints/C_seed42_best.pt, /kaggle/input/**/v25_checkpoints/C_seed42_best.pt, ...) -> RETRAINING with v22's train_one_run
[C seed=42 HTT-time ON / LACA-lag ON] HTT_TIME_EMBED_ENABLED = True | LACA_LAG_BIAS_ENABLED = True (beta 1.0) — read from the constructed model | fusion MambaBlock (874,496 params)
[C seed=42 HTT-time ON / LACA-lag ON] [PAIRING] initial weights 5850841e699f vs v18 5850841e699f | RNG 5641b401720d vs v18 5641b401720d -> ✅ identical start and RNG stream (same in all 4 conditions)
[C seed=42 HTT-time ON / LACA-lag ON] [SWITCH CHECK before training] HTT_TIME_EMBED_ENABLED=True: time-embedding term max audio 5.743e-01, sensor 6.356e-01 | output change for tau+0.3: audio 7.920e-01, sensor 8.713e-01
[C seed=42 HTT-time ON / LACA-lag ON] [SWITCH CHECK before training] LACA_LAG_BIAS_ENABLED=True (beta read back: 1.0 / 1.0): max|beta * LN(g(Δτ))| = 0.0
[C seed

C seed 42 epoch 01/20:   0%|          | 0/155 [00:00<?, ?it/s]

[C seed=42 HTT-time ON / LACA-lag ON] epoch 01/20 | lag term 2.3752 | train loss 1.4174 (cls 1.0706, align 3.1030, cons 0.3650) | val macro-F1 0.8287  * best


C seed 42 epoch 02/20:   0%|          | 0/155 [00:00<?, ?it/s]

[C seed=42 HTT-time ON / LACA-lag ON] epoch 02/20 | lag term 2.2236 | train loss 0.9226 (cls 0.5944, align 2.6677, cons 0.6139) | val macro-F1 0.7019


...

[D seed=99 HTT-time OFF / LACA-lag OFF] epoch 18/20 | lag term 0.0000 (exactly 0 ✅) | train loss 0.3446 (cls 0.1764, align 1.2536, cons 0.4287) | val macro-F1 0.8553


D seed 99 epoch 19/20:   0%|          | 0/155 [00:00<?, ?it/s]

[D seed=99 HTT-time OFF / LACA-lag OFF] epoch 19/20 | lag term 0.0000 (exactly 0 ✅) | train loss 0.3119 (cls 0.1435, align 1.2520, cons 0.4318) | val macro-F1 0.8896


D seed 99 epoch 20/20:   0%|          | 0/155 [00:00<?, ?it/s]

[D seed=99 HTT-time OFF / LACA-lag OFF] epoch 20/20 | lag term 0.0000 (exactly 0 ✅) | train loss 0.3163 (cls 0.1487, align 1.2431, cons 0.4326) | val macro-F1 0.8921
[D seed=99 HTT-time OFF / LACA-lag OFF] [SWITCH CHECK best checkpoint, epoch 16] HTT_TIME_EMBED_ENABLED=False: time-embedding term max audio 0.000e+00, sensor 0.000e+00 | output change for tau+0.3: audio 0.000e+00, sensor 0.000e+00
[D seed=99 HTT-time OFF / LACA-lag OFF] [SWITCH CHECK best checkpoint, epoch 16] LACA_LAG_BIAS_ENABLED=False (beta read back: 0.0 / 0.0): max|beta * LN(g(Δτ))| = 0.0
[D seed=99 HTT-time OFF / LACA-lag OFF] [SWITCH CHECK best checkpoint, epoch 16] ✅ HTT time OFF: its contribution is EXACTLY 0 and HTT ignores tau entirely
[D seed=99 HTT-time OFF / LACA-lag OFF] [SWITCH CHECK best checkpoint, epoch 16] ✅ LACA lag OFF: the lag term added to the attention scores is EXACTLY 0
[D seed=99] [REPRODUCTION vs v22] accuracy 0.8761 (v22 0.8761, +0.0000) | macro_f1 0.8706 (v22 0.8706, +0.0000) | balanced_accu

## 8. Part 3 — C − D at every misalignment level, trend, and verdict
Each arm and type gets a table. For each magnitude it lists C and D (mean ± SD over seeds) and the C − D gap: mean,
SD, wins out of 5 and paired-t p.

**Trend.** Two per-seed measures are summarised as mean, SD, wins and t p:
- **Δgap** = (C − D) at the largest magnitude minus (C − D) at 0;
- the per-seed least-squares slope of the gap against magnitude.

**Verdict** for each arm and type:
- **GROWS (real)** when mean Δgap > SD(Δgap) > 0, and mean C − D at the largest magnitude is > 0 and above its own SD.
- **GROWS, but C is still not ahead** when only the first part holds.
- **DECREASES** when mean Δgap < 0 and |mean| > SD.
- **FLAT** otherwise.

**Secondary.**
- Both curves are shown for every arm and type.
- D's flatness under Arm 1 is checked against the bit-identity results.
- C's degradation is reported as C at max − C at 0, paired over seeds.
- Arm 1's verdict is constrained by construction (see the header). Arm 2 is the one that answers the research question.

In [8]:
# ==============================================================================
# PART 3 — paired C - D per (arm, type, magnitude), trend, verdict; secondary C / D curves
# ==============================================================================
from scipy import stats


def _get(arm, kind, mag, cond, seed, key="macro_f1"):
    """kind 'none' / mag 0 = the arm's own baseline (Arm 1: the shared clean result)."""
    if mag == 0:
        kind = "none"
    hits = [r for r in RESULTS if r["arm"] == arm and r["perturbation"] == kind and r["magnitude"] == mag
            and r["condition"] == cond and r["seed"] == seed]
    assert len(hits) == 1, (arm, kind, mag, cond, seed, len(hits))
    return hits[0][key]


def _pstats(d):
    d = np.asarray(d, dtype=float)
    sd = float(d.std(ddof=1))
    p = (1.0 if np.all(d == 0) else 0.0) if d.std() == 0 else float(stats.ttest_1samp(d, 0.0).pvalue)
    return {"d": d, "mean": float(d.mean()), "sd": sd, "wins": int((d > 0).sum()), "p": p}


SWEEPS = {"delay": DELAYS, "jitter": JITTER_SIGMAS}
VERDICTS, GAPS = {}, {}
for arm in ARMS:
    for kind, mags in SWEEPS.items():
        print("\n" + "=" * 118)
        print(f"ARM {arm.upper()} | TYPE {kind.upper()} ({'fixed delay δ' if kind == 'delay' else 'per-window jitter σ'}) | "
              f"macro-F1, 5 seeds" + (" | subset of " + str(len(CONTENT_IDX)) + " windows" if arm == "content_shift" else ""))
        print("=" * 118)
        print(f"    {'magnitude':>9}{'C mean':>9}{'C SD':>8}{'D mean':>9}{'D SD':>8}{'C-D mean':>10}{'C-D SD':>8}{'wins':>6}{'t p':>8}"
              f"   per-seed C-D ({', '.join(map(str, SEEDS))})")
        for mag in mags:
            c = [_get(arm, kind, mag, "C", s) for s in SEEDS]
            d = [_get(arm, kind, mag, "D", s) for s in SEEDS]
            g = GAPS[(arm, kind, mag)] = _pstats(np.array(c) - np.array(d))
            print(f"    {mag:>9.2f}{np.mean(c):>9.4f}{np.std(c, ddof=1):>8.4f}{np.mean(d):>9.4f}{np.std(d, ddof=1):>8.4f}"
                  f"{g['mean']:>+10.4f}{g['sd']:>8.4f}{g['wins']:>4}/5{g['p']:>8.4f}   " + ", ".join(f"{x:+.4f}" for x in g["d"]))
        lo, hi = mags[0], mags[-1]
        dgap = _pstats(GAPS[(arm, kind, hi)]["d"] - GAPS[(arm, kind, lo)]["d"])
        slopes = _pstats([np.polyfit(mags, [GAPS[(arm, kind, m)]["d"][i] for m in mags], 1)[0] for i in range(len(SEEDS))])
        c_deg = _pstats([_get(arm, kind, hi, "C", s) - _get(arm, kind, lo, "C", s) for s in SEEDS])
        d_deg = _pstats([_get(arm, kind, hi, "D", s) - _get(arm, kind, lo, "D", s) for s in SEEDS])
        g_hi = GAPS[(arm, kind, hi)]
        if dgap["mean"] > 0 and dgap["mean"] > dgap["sd"]:
            verdict = ("GROWS — REAL: C's advantage over D increases with misalignment and C is clearly ahead at the largest "
                       "magnitude" if g_hi["mean"] > 0 and g_hi["mean"] > g_hi["sd"] else
                       "GROWS, but C is NOT clearly ahead at the largest magnitude (the gap moves in C's favour, e.g. D's "
                       "lead shrinks, without C overtaking D beyond noise)")
        elif dgap["mean"] < 0 and abs(dgap["mean"]) > dgap["sd"]:
            verdict = "DECREASES: C's relative standing WORSENS as misalignment grows"
        else:
            verdict = "FLAT: no change in C's relative standing beyond the paired noise"
        VERDICTS[(arm, kind)] = {"verdict": verdict, "dgap": dgap, "slope": slopes, "gap0": GAPS[(arm, kind, lo)],
                                 "gap_hi": g_hi, "c_deg": c_deg, "d_deg": d_deg}
        print(f"    TREND  Δgap = (C-D)@{hi} - (C-D)@{lo}: mean {dgap['mean']:+.4f}, SD {dgap['sd']:.4f}, wins {dgap['wins']}/5, "
              f"t p {dgap['p']:.4f} | slope of C-D per unit magnitude: mean {slopes['mean']:+.4f}, SD {slopes['sd']:.4f}")
        print(f"    CURVES C@{hi} - C@{lo}: {c_deg['mean']:+.4f} (SD {c_deg['sd']:.4f}) | D@{hi} - D@{lo}: {d_deg['mean']:+.4f} "
              f"(SD {d_deg['sd']:.4f})")
        print(f"    -> {verdict}")

print("\n[CLASS 4 RECALL] mean over seeds, C / D, per arm / type / magnitude:")
for arm in ARMS:
    for kind, mags in SWEEPS.items():
        print(f"    {arm:<14}{kind:<7}" + " | ".join(
            f"{m:.2f}: {np.mean([_get(arm, kind, m, 'C', s, 'class4_recall') for s in SEEDS]):.4f} / "
            f"{np.mean([_get(arm, kind, m, 'D', s, 'class4_recall') for s in SEEDS]):.4f}" for m in mags))

TAU_INV_ALL = bool(TAU_INVARIANCE) and all(TAU_INVARIANCE.values())
print("\n" + "#" * 118)
print("#  ANSWER — does the temporal mechanism's advantage (C - D, macro-F1) grow as misalignment increases?")
print("#" * 118)
for arm in ARMS:
    for kind in SWEEPS:
        v = VERDICTS[(arm, kind)]
        print(f"#  {arm:<14} {kind:<7}: C-D at 0 {v['gap0']['mean']:+.4f} (SD {v['gap0']['sd']:.4f}) -> at max "
              f"{v['gap_hi']['mean']:+.4f} (SD {v['gap_hi']['sd']:.4f}) | Δgap {v['dgap']['mean']:+.4f} (SD {v['dgap']['sd']:.4f}) "
              f"-> {v['verdict'].split(':')[0].split(' —')[0]}")
print("#")
print("#  Arm 1 (tau-only) is constrained by construction: D is " + ("exactly flat (bit-identical at every magnitude) ✅"
      if TAU_INV_ALL else "🚨 NOT flat — injection bug, see above") + ", and only C's inputs change. Its C - D gap can "
      "only move with C's own degradation, so it cannot show a mechanism that pays off under real misalignment.")
v_d, v_j = VERDICTS[("content_shift", "delay")], VERDICTS[("content_shift", "jitter")]
print(f"#  Arm 2 (content shift: real misalignment, honest timestamps) answers the research question:")
print(f"#     fixed delay : {v_d['verdict']}")
print(f"#     jitter      : {v_j['verdict']}")
if any(VERDICTS[("content_shift", k)]["verdict"].startswith("GROWS — REAL") for k in SWEEPS):
    print("#  -> The temporal mechanism's value IS conditional on misalignment: it shows up once the modalities are really "
          "misaligned, which the clean TII data lacks.")
else:
    print("#  -> No reliable conditional advantage: even under real misalignment with true timestamps, C does not gain "
          "ground on D beyond the paired noise. (C was trained only on aligned data. A model trained WITH misalignment "
          "might behave differently; that is a separate experiment.)")
print("#" * 118)



ARM TAU_ONLY | TYPE DELAY (fixed delay δ) | macro-F1, 5 seeds
    magnitude   C mean    C SD   D mean    D SD  C-D mean  C-D SD  wins     t p   per-seed C-D (42, 123, 2024, 7, 99)
         0.00   0.8796  0.0141   0.8876  0.0136   -0.0080  0.0097   1/5  0.1392   -0.0164, -0.0110, -0.0082, +0.0085, -0.0129
         0.05   0.8800  0.0112   0.8876  0.0136   -0.0075  0.0096   1/5  0.1540   -0.0189, -0.0108, -0.0082, +0.0075, -0.0073
         0.10   0.8786  0.0120   0.8876  0.0136   -0.0089  0.0145   1/5  0.2392   -0.0216, -0.0225, -0.0088, +0.0128, -0.0046
         0.20   0.8790  0.0092   0.8876  0.0136   -0.0085  0.0134   1/5  0.2282   -0.0224, -0.0227, -0.0024, +0.0074, -0.0026
         0.30   0.8790  0.0103   0.8876  0.0136   -0.0085  0.0138   1/5  0.2389   -0.0262, -0.0171, -0.0031, +0.0099, -0.0062
    TREND  Δgap = (C-D)@0.3 - (C-D)@0.0: mean -0.0006, SD 0.0072, wins 3/5, t p 0.8719 | slope of C-D per unit magnitude: mean -0.0024, SD 0.0249
    CURVES C@0.3 - C@0.0: -0.0006 (SD 0.007

## 9. Final validation
A recap of every check:
- Part 1: reuse or retrain, and reproduction of v22;
- the evaluation path matching v22's test loader;
- H2 reproduction for seed 42;
- D's τ-invariance under both Arm-1 types;
- identical jitter across models;
- the content-shift re-slicer's exact match at shift 0.

In [9]:
# ==============================================================================
# FINAL VALIDATION — every verification hook in one place
# ==============================================================================
print("=" * 100)
print("FINAL VALIDATION")
print("=" * 100)
print("  Part 1 (per model): " + ", ".join(f"{c}{s}: {MODEL_SOURCES[(c, s)]}" for s, c in RUN_ORDER))
print("  Part 1 reproduction vs v22 (4 metrics ±0.0005 + identical confusion matrix):")
for s, c in RUN_ORDER:
    d, cm, ok = REPRO[(c, s)]
    print(f"      {c} seed {s:<5}: max |delta| {d:.1e}, confusion matrix {'identical' if cm else 'DIFFERENT'} -> "
          + ("✅" if ok else "🚨 does NOT reproduce v22"))
_jit_rows = defaultdict(set)
for r in RESULTS:
    if r["perturbation"] == "jitter":
        _jit_rows[(r["arm"], r["magnitude"])].add(r["shift_sha1"])
checks = {
    "every model reproduces v22's clean result": all(ok for _, _, ok in REPRO.values()),
    "eval path == v22 test_loader (bit-identical), every model": all(LOADER_MATCH.values()),
    "seed-42 tau-only delays reproduce v22's H2 (C and D, ±0.0005)": bool(H2_REPRO) and all(v <= SAME_SEED_TOLERANCE for v in H2_REPRO.values()),
    "D tau-invariant under Arm-1 DELAY at every magnitude, every seed": all(v for (s, k, m), v in TAU_INVARIANCE.items() if k == "delay"),
    "D tau-invariant under Arm-1 JITTER at every magnitude, every seed": all(v for (s, k, m), v in TAU_INVARIANCE.items() if k == "jitter"),
    f"jitter noise identical across all models (seed {JITTER_SEED})": all(len(v) == 1 for v in _jit_rows.values()),
    "content-shift re-slicer == v18 test windows at shift 0": CONTENT_BASELINE_EXACT,
    f"evaluations logged ({len(RUN_ORDER)} models x {2 + 2 * len(PERTURBATIONS)})": len(RESULTS) == len(RUN_ORDER) * (2 + 2 * len(PERTURBATIONS)),
    "fusion backbone is real mamba_ssm": MAMBA_CLASS is not None,
}
for name, ok in checks.items():
    print(f"  {name:<66}: {'✅ pass' if ok else '🚨 FAIL'}")
print(f"  {'jitter shift sha1 per sigma (same for every model and both arms)':<66}: "
      + ", ".join(f"{s}: {h}" for s, h in JITTER_SHA1.items()))
print(f"  {'Arm-2 evaluation subset':<66}: {len(CONTENT_IDX)} of {N_TEST} test windows")
print("=" * 100)
if not checks["D tau-invariant under Arm-1 DELAY at every magnitude, every seed"] or \
        not checks["D tau-invariant under Arm-1 JITTER at every magnitude, every seed"]:
    print("🚨🚨🚨 Condition D changed under tau-only misalignment — this is an INJECTION BUG, not a finding. Do not use Arm 1.")
if not all(checks.values()):
    print("🚨 at least one check FAILED — read the flagged line before using the verdict.")


FINAL VALIDATION
  Part 1 (per model): C42: retrained in v25, D42: retrained in v25, C123: retrained in v25, D123: retrained in v25, C2024: retrained in v25, D2024: retrained in v25, C7: retrained in v25, D7: retrained in v25, C99: retrained in v25, D99: retrained in v25
  Part 1 reproduction vs v22 (4 metrics ±0.0005 + identical confusion matrix):
      C seed 42   : max |delta| 0.0e+00, confusion matrix identical -> ✅
      D seed 42   : max |delta| 0.0e+00, confusion matrix identical -> ✅
      C seed 123  : max |delta| 0.0e+00, confusion matrix identical -> ✅
      D seed 123  : max |delta| 0.0e+00, confusion matrix identical -> ✅
      C seed 2024 : max |delta| 9.7e-07, confusion matrix identical -> ✅
      D seed 2024 : max |delta| 0.0e+00, confusion matrix identical -> ✅
      C seed 7    : max |delta| 3.0e-06, confusion matrix identical -> ✅
      D seed 7    : max |delta| 0.0e+00, confusion matrix identical -> ✅
      C seed 99   : max |delta| 0.0e+00, confusion matrix identic